In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(
    llm_name
)

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM loaded")
print("Model:", llm_name)
print("Device:", llm_model.device)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

c:\Projects\DocuMind\ml\.venv\Lib\site-packages\huggingface_hub\file_download.py:150: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sudee\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded
Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0


In [2]:
messages = [
    {
        "role": "system",
        "content": "You are a helpful document assistant."
    },
    {
        "role": "user",
        "content": "What is an invoice? Answer in one sentence."
    }
]

text = llm_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = llm_tokenizer(
    text,
    return_tensors="pt"
).to(llm_model.device)

with torch.no_grad():
    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

answer = llm_tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print(answer)

An invoice is a legal document issued by a seller to a buyer, listing the goods or services provided and the amount due.


In [3]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path

from sentence_transformers import SentenceTransformer

search_artifacts = Path("../models/hybrid_search")

# Load chunks
chunks_df = pd.read_parquet(
    search_artifacts / "chunks.parquet"
)

# Load embeddings
embeddings = np.load(
    search_artifacts / "embeddings.npy"
)

# Load BM25
with open(
    search_artifacts / "bm25.pkl",
    "rb"
) as f:
    bm25 = pickle.load(f)

# Load embedding model
embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("RAG retrieval components loaded")
print("Chunks:", len(chunks_df))
print("Embeddings:", embeddings.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RAG retrieval components loaded
Chunks: 16851
Embeddings: (16851, 384)


In [4]:
def hybrid_search_rrf(query, top_k=5, candidate_k=20, k=60):
    # BM25
    query_tokens = query.lower().split()

    bm25_scores = bm25.get_scores(
        query_tokens
    )

    bm25_ranked = np.argsort(
        bm25_scores
    )[::-1][:candidate_k]

    # Semantic search
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    semantic_scores = embeddings @ query_embedding

    semantic_ranked = np.argsort(
        semantic_scores
    )[::-1][:candidate_k]

    # Reciprocal Rank Fusion
    rrf_scores = {}

    for rank, idx in enumerate(bm25_ranked):
        rrf_scores[idx] = (
            rrf_scores.get(idx, 0)
            + 1 / (k + rank + 1)
        )

    for rank, idx in enumerate(semantic_ranked):
        rrf_scores[idx] = (
            rrf_scores.get(idx, 0)
            + 1 / (k + rank + 1)
        )

    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    results = chunks_df.iloc[
        ranked_indices
    ].copy()

    results["rrf_score"] = [
        rrf_scores[idx]
        for idx in ranked_indices
    ]

    return results.reset_index(drop=True)


print("RRF retrieval ready")

RRF retrieval ready


In [5]:
query = "Which contracts mention early termination penalties?"

results = hybrid_search_rrf(
    query,
    top_k=5
)

display(
    results[
        [
            "document_id",
            "label",
            "rrf_score",
            "text"
        ]
    ]
)

,document_id,label,rrf_score,text
0,contract_0441,Contract,0.031010,(30) days from receipt of notice to cure such ...
1,contract_0356,Contract,0.030478,of the Agreement by both parties and shall ter...
2,contract_0437,Contract,0.030366,Obligations 17 18.2. Exceptions to Obligations...
3,email_1065,Email,0.028992,Termination Notice To Accenture: As discussed ...
4,contract_0446,Contract,0.027810,B's personnel. Such payment shall be deducted ...


In [6]:
def build_rag_prompt(query, results):
    context_parts = []

    for i, row in results.iterrows():
        context_parts.append(
            f"[Source {i + 1} | Document: {row['document_id']} | "
            f"Type: {row['label']}]\n"
            f"{row['text']}"
        )

    context = "\n\n".join(context_parts)

    prompt = f"""
You are DocuMind, an enterprise document assistant.

Answer the user's question using ONLY the provided document context.

Rules:
- Do not use outside knowledge.
- If the context does not contain enough information, say so.
- Cite the source document IDs in your answer.
- Do not invent facts.

DOCUMENT CONTEXT:
{context}

USER QUESTION:
{query}
"""

    return prompt


query = "Which contracts mention early termination penalties?"

results = hybrid_search_rrf(
    query,
    top_k=5
)

rag_prompt = build_rag_prompt(
    query,
    results
)

print(rag_prompt)


You are DocuMind, an enterprise document assistant.

Answer the user's question using ONLY the provided document context.

Rules:
- Do not use outside knowledge.
- If the context does not contain enough information, say so.
- Cite the source document IDs in your answer.
- Do not invent facts.

DOCUMENT CONTEXT:
[Source 1 | Document: contract_0441 | Type: Contract]
(30) days from receipt of notice to cure such breach, except for nonpayment by Customer, which must be cured within seven (7) business days from receipt of notice. If such breach has not been timely cured, then the non-breaching party may immediately terminate this Agreement upon written notice; provided, however, it is understood that in the event IBM has so breached this Agreement IBM shall not be entitled to recover the early termination charges described in Section 3.4(b) below. 3.4 Termination for Convenience Customer may terminate this Agreement (including all Service Option Attachments) or any Service Option Attachmen

In [7]:
def generate_rag_answer(query, top_k=5):

    # Retrieve relevant chunks
    results = hybrid_search_rrf(
        query,
        top_k=top_k
    )

    # Build grounded context
    context_parts = []

    for i, row in results.iterrows():

        context_parts.append(
            f"[Source {i + 1} | "
            f"Document: {row['document_id']} | "
            f"Type: {row['label']}]\n"
            f"{row['text']}"
        )

    context = "\n\n".join(context_parts)

    # Grounded prompt
    messages = [
        {
            "role": "system",
            "content": (
                "You are DocuMind, an enterprise document assistant. "
                "Answer ONLY from the provided document context. "
                "Do not use outside knowledge. "
                "If the context is insufficient, say so. "
                "Always cite the document IDs that support your answer."
            )
        },
        {
            "role": "user",
            "content": (
                f"DOCUMENT CONTEXT:\n\n"
                f"{context}\n\n"
                f"QUESTION:\n{query}"
            )
        }
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer, results


print("RAG answer generator ready")

RAG answer generator ready


In [8]:
query = "Which contracts mention early termination penalties?"

answer, retrieved_results = generate_rag_answer(
    query,
    top_k=5
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nSOURCES:")
for _, row in retrieved_results.iterrows():
    print(
        f"- {row['document_id']} ({row['label']})"
    )

QUESTION:
Which contracts mention early termination penalties?

ANSWER:
Based on the information provided in the given documents, there are two contracts that mention early termination penalties:

1. **Contract 0441** mentions "early termination charges" in Section 3.4(b). This section states that if IBM breaches the agreement, it cannot recover the early termination charges specified in Attachment A and applicable Service Option Attachments.

2. **Contract 0446** includes provisions related to early termination penalties. Specifically, Article 10 states that Party B must obtain Party A's written consent before terminating the Agreement. Additionally, if Party B terminates the Agreement without consent, Party B must pay one-month freight as liquidated damages. Furthermore, within the contract period, Party B cannot charge the freight difference if Party A rents the same-level vehicles, and Party B must also compensate Party A's other losses.

These clauses indicate that both contracts 

In [9]:
def hybrid_search_rrf(
    query,
    top_k=5,
    candidate_k=30,
    k=60,
    label_filter=None
):
    # ---------------------------------------------
    # Optional document-type filtering
    # ---------------------------------------------
    if label_filter is not None:
        allowed_mask = (
            chunks_df["label"].values == label_filter
        )
    else:
        allowed_mask = np.ones(
            len(chunks_df),
            dtype=bool
        )

    # ---------------------------------------------
    # BM25
    # ---------------------------------------------
    query_tokens = query.lower().split()

    bm25_scores = bm25.get_scores(
        query_tokens
    )

    bm25_scores = np.asarray(
        bm25_scores,
        dtype=np.float32
    )

    bm25_scores[~allowed_mask] = -np.inf

    bm25_ranked = np.argsort(
        bm25_scores
    )[::-1][:candidate_k]

    # ---------------------------------------------
    # Semantic search
    # ---------------------------------------------
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    semantic_scores = embeddings @ query_embedding

    semantic_scores = np.asarray(
        semantic_scores,
        dtype=np.float32
    )

    semantic_scores[~allowed_mask] = -np.inf

    semantic_ranked = np.argsort(
        semantic_scores
    )[::-1][:candidate_k]

    # ---------------------------------------------
    # RRF
    # ---------------------------------------------
    rrf_scores = {}

    for rank, idx in enumerate(bm25_ranked):
        if np.isfinite(bm25_scores[idx]):
            rrf_scores[idx] = (
                rrf_scores.get(idx, 0)
                + 1 / (k + rank + 1)
            )

    for rank, idx in enumerate(semantic_ranked):
        if np.isfinite(semantic_scores[idx]):
            rrf_scores[idx] = (
                rrf_scores.get(idx, 0)
                + 1 / (k + rank + 1)
            )

    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    results = chunks_df.iloc[
        ranked_indices
    ].copy()

    results["rrf_score"] = [
        rrf_scores[idx]
        for idx in ranked_indices
    ]

    return results.reset_index(drop=True)

In [10]:
query = "Which contracts mention early termination penalties?"

results = hybrid_search_rrf(
    query,
    top_k=5,
    label_filter="Contract"
)

display(
    results[
        [
            "document_id",
            "label",
            "rrf_score",
            "text"
        ]
    ]
)

,document_id,label,rrf_score,text
0,contract_0356,Contract,0.031099,of the Agreement by both parties and shall ter...
1,contract_0441,Contract,0.031010,(30) days from receipt of notice to cure such ...
2,contract_0437,Contract,0.030622,Obligations 17 18.2. Exceptions to Obligations...
3,contract_0320,Contract,0.028589,"by law, government regulation, or court order...."
4,contract_0446,Contract,0.027972,B's personnel. Such payment shall be deducted ...


In [11]:
def generate_rag_answer(
    query,
    top_k=5,
    label_filter=None
):
    # -------------------------------------------------
    # Retrieve relevant chunks
    # -------------------------------------------------
    results = hybrid_search_rrf(
        query,
        top_k=top_k,
        label_filter=label_filter
    )

    # -------------------------------------------------
    # Build context
    # -------------------------------------------------
    context_parts = []

    for i, row in results.iterrows():

        context_parts.append(
            f"[Source {i + 1} | "
            f"Document: {row['document_id']} | "
            f"Type: {row['label']}]\n"
            f"{row['text']}"
        )

    context = "\n\n".join(context_parts)

    # -------------------------------------------------
    # Grounded prompt
    # -------------------------------------------------
    messages = [
        {
            "role": "system",
            "content": (
                "You are DocuMind, an enterprise document assistant.\n"
                "Answer ONLY using the provided document context.\n"
                "Do not use outside knowledge.\n"
                "If the context is insufficient, explicitly say so.\n"
                "Only claim information directly supported by the context.\n"
                "Cite the document IDs supporting each important claim."
            )
        },
        {
            "role": "user",
            "content": (
                f"DOCUMENT CONTEXT:\n\n"
                f"{context}\n\n"
                f"QUESTION:\n{query}"
            )
        }
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer, results


print("Updated RAG generator ready")

Updated RAG generator ready


In [12]:
query = "Which contracts mention early termination penalties?"

answer, results = generate_rag_answer(
    query,
    top_k=5,
    label_filter="Contract"
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nRETRIEVED SOURCES:")
for _, row in results.iterrows():
    print(
        f"- {row['document_id']} ({row['label']})"
    )

QUESTION:
Which contracts mention early termination penalties?

ANSWER:
Based on the provided documents, there are two contracts that mention early termination penalties:

1. **Contract ID: contract_0441**
   - Clause 3.4 states: "Customer may terminate this Agreement (including all Service Option Attachments) or any Service Option Attachment (with the exception of any Service Option Attachment that is a prerequisite for the provision of Services under a non-terminated Service Option Attachment) for convenience at the end of any calendar month by: a. providing at least one month's prior written notice to IBM; and b. paying the applicable early termination charges, if any, specified in Attachment A and applicable Service Option Attachments."

2. **Contract ID: contract_0437**
   - Clause 19.5 specifies: "Early Termination by the Licensee: The Licensee may terminate this Agreement for convenience at any time during the term of this Agreement by giving the Licensor written notice of termi

In [13]:
def expand_retrieved_context(
    results,
    neighbor_chunks=2
):
    """
    For every retrieved chunk, include nearby chunks
    from the same document.

    Example:
        retrieved chunk 5
        -> include chunks 3, 4, 5, 6, 7
    """

    selected = {}

    for _, row in results.iterrows():

        document_id = row["document_id"]
        center_index = int(row["chunk_index"])

        # Find nearby chunks from the same document
        nearby = chunks_df[
            chunks_df["document_id"] == document_id
        ]

        nearby = nearby[
            (
                nearby["chunk_index"]
                >= center_index - neighbor_chunks
            )
            &
            (
                nearby["chunk_index"]
                <= center_index + neighbor_chunks
            )
        ]

        for _, chunk in nearby.iterrows():

            key = (
                chunk["document_id"],
                int(chunk["chunk_index"])
            )

            if key not in selected:
                selected[key] = chunk

    expanded = pd.DataFrame(
        list(selected.values())
    )

    # Keep document/chunk order for readability
    expanded = expanded.sort_values(
        ["document_id", "chunk_index"]
    ).reset_index(drop=True)

    return expanded


print("Context expansion ready")

Context expansion ready


In [14]:
query = "Which contracts mention early termination penalties?"

results = hybrid_search_rrf(
    query,
    top_k=5,
    label_filter="Contract"
)

expanded_results = expand_retrieved_context(
    results,
    neighbor_chunks=2
)

print("Original retrieved chunks:", len(results))
print("Expanded context chunks:", len(expanded_results))

display(
    expanded_results[
        [
            "document_id",
            "chunk_index",
            "label",
            "text"
        ]
    ]
)

Original retrieved chunks: 5
Expanded context chunks: 18


,document_id,chunk_index,label,text
0,contract_0320,2,Contract,implementation issues. SIC shall allow TKCI to...
1,contract_0320,3,Contract,and management services and capital resources ...
2,contract_0320,4,Contract,"by law, government regulation, or court order...."
3,contract_0356,0,Contract,Exhibit 10.12 Certain identified information h...
4,contract_0356,1,Contract,"(SOPs) or other guidelines as provided, which ..."
5,contract_0356,2,Contract,of the Agreement by both parties and shall ter...
6,contract_0356,3,Contract,for bona fide reconstruction) or has a receive...
7,contract_0356,4,Contract,"terms of this Agreement; provided, however, th..."
8,contract_0437,0,Contract,Exhibit 10.14 License and Development Agreemen...
9,contract_0437,1,Contract,Obligations 17 18.2. Exceptions to Obligations...


In [16]:
import re

# Documents retrieved for the query
retrieved_doc_ids = results["document_id"].unique().tolist()

evidence_terms = [
    "termination",
    "early termination",
    "termination fee",
    "termination charges",
    "liquidated damages",
    "penalty"
]

evidence_chunks = chunks_df[
    chunks_df["document_id"].isin(retrieved_doc_ids)
].copy()

pattern = "|".join(
    re.escape(term)
    for term in evidence_terms
)

evidence_chunks = evidence_chunks[
    evidence_chunks["text"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )
].copy()

print("Evidence chunks found:", len(evidence_chunks))

display(
    evidence_chunks[
        ["document_id", "chunk_index", "label", "text"]
    ]
)

Evidence chunks found: 12


,document_id,chunk_index,label,text
4418,contract_0320,0,Contract,1. 2. 2.1 2.2 3. 3.1 3.2 4. 4.1 EXHIBIT 1.1 St...
4421,contract_0320,3,Contract,and management services and capital resources ...
4422,contract_0320,4,Contract,"by law, government regulation, or court order...."
8974,contract_0446,2,Contract,A by delivery delay due to vehicles detention ...
8975,contract_0446,3,Contract,"guarantee the authenticity, completeness, lega..."
8976,contract_0446,4,Contract,B's personnel. Such payment shall be deducted ...
9247,contract_0356,1,Contract,"(SOPs) or other guidelines as provided, which ..."
9248,contract_0356,2,Contract,of the Agreement by both parties and shall ter...
12645,contract_0441,2,Contract,"or using the Services, or the results or produ..."
12646,contract_0441,3,Contract,(30) days from receipt of notice to cure such ...


In [1]:
import torch
import numpy as np
import pandas as pd
import bm25s
from pathlib import Path
from sentence_transformers import SentenceTransformer

processed_dir = Path("../processed")

# Full-text chunks
full_chunks_df = pd.read_parquet(
    processed_dir / "full_text_chunks.parquet",
    columns=[
        "chunk_id",
        "document_id",
        "label",
        "source",
        "chunk_index",
        "text"
    ]
)

# Full-text BGE embeddings
full_embeddings = np.load(
    processed_dir / "full_text_embeddings.npy",
    mmap_mode="r"
)

# Full-text BM25S index
full_bm25s = bm25s.BM25.load(
    str(processed_dir / "full_text_bm25s"),
    load_corpus=False
)

# Semantic model
embedding_model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("Full-text RAG index loaded.")
print("Chunks:", len(full_chunks_df))
print("Embeddings:", full_embeddings.shape)
print("Embedding device:", embedding_model.device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Full-text RAG index loaded.
Chunks: 223234
Embeddings: (223234, 384)
Embedding device: cuda:0


In [2]:
def hybrid_search_full_rag(
    query,
    top_k=5,
    candidate_k=30,
    rrf_k=60,
    label_filter=None
):
    # -------------------------------------------------
    # Document-type filter
    # -------------------------------------------------
    if label_filter is not None:
        allowed_mask = (
            full_chunks_df["label"].values == label_filter
        )
    else:
        allowed_mask = np.ones(
            len(full_chunks_df),
            dtype=bool
        )

    # -------------------------------------------------
    # BM25S retrieval
    # -------------------------------------------------
    query_tokens = bm25s.tokenize(
        [query],
        stopwords="en"
    )

    bm25_results, _ = full_bm25s.retrieve(
        query_tokens,
        k=candidate_k,
        show_progress=False
    )

    bm25_indices = np.asarray(
        bm25_results[0],
        dtype=int
    )

    bm25_indices = [
        int(idx)
        for idx in bm25_indices
        if allowed_mask[idx]
    ][:candidate_k]

    # -------------------------------------------------
    # Semantic retrieval
    # -------------------------------------------------
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )[0]

    semantic_scores = full_embeddings @ query_embedding

    semantic_scores = np.asarray(
        semantic_scores,
        dtype=np.float32
    )

    semantic_scores[~allowed_mask] = -np.inf

    semantic_indices = np.argsort(
        semantic_scores
    )[::-1][:candidate_k]

    # -------------------------------------------------
    # Reciprocal Rank Fusion
    # -------------------------------------------------
    rrf_scores = {}

    for rank, idx in enumerate(bm25_indices):
        rrf_scores[idx] = (
            rrf_scores.get(idx, 0.0)
            + 1.0 / (rrf_k + rank + 1)
        )

    for rank, idx in enumerate(semantic_indices):
        rrf_scores[int(idx)] = (
            rrf_scores.get(int(idx), 0.0)
            + 1.0 / (rrf_k + rank + 1)
        )

    ranked_indices = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True
    )[:top_k]

    results = full_chunks_df.iloc[
        ranked_indices
    ].copy()

    results["rrf_score"] = [
        rrf_scores[idx]
        for idx in ranked_indices
    ]

    results["semantic_score"] = [
        semantic_scores[idx]
        for idx in ranked_indices
    ]

    return results.reset_index(drop=True)


print("Full-text RAG retriever ready")

Full-text RAG retriever ready


In [3]:
query = "Which contracts mention early termination penalties?"

results = hybrid_search_full_rag(
    query,
    top_k=5,
    candidate_k=30,
    label_filter="Contract"
)

for i, row in results.iterrows():
    print("\n" + "=" * 100)
    print(
        f"Rank: {i + 1}\n"
        f"Document: {row['document_id']}\n"
        f"Chunk: {row['chunk_index']}\n"
        f"RRF score: {row['rrf_score']:.6f}"
    )
    print("\n" + row["text"])

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]


Rank: 1
Document: contract_0112
Chunk: 2
RRF score: 0.016393

notice if the other party: (w) becomes insolvent; (x) files a petition in bankruptcy; (y) makes an assignment for the benefit of its creditors; or (z) breach any of its obligations under this Agreement in any material respect, which breach is not remedied within thirty (30) days following written notice to such party. EFFECT OF TERMINATION: Any termination shall be without any liability or obligation of the terminating party, other than with respect to any breach of this Agreement prior to termination. The provisions relating to property rights and confidentiality shall survive any termination or expiration of this Agreement. All revenue sharing ceases with the termination of this Agreement. Initialed THE HENRY FILM AND ENTERTAINMENT CORPORATION:______ Initialed PACIFICAP ENTERTAINMENT:______ Page 3 of 6 Source: PACIFICAP ENTERTAINMENT HOLDINGS INC, 8-K/A, 11/15/2005 PACIFICAP ENTERTAINMENT Agreement with THE HENRY FILM AND

In [4]:
def generate_full_rag_answer(
    query,
    top_k=5,
    label_filter=None
):
    # ---------------------------------------------
    # Retrieve from full-text hybrid index
    # ---------------------------------------------
    results = hybrid_search_full_rag(
        query,
        top_k=top_k,
        candidate_k=30,
        label_filter=label_filter
    )

    # ---------------------------------------------
    # Build grounded context
    # ---------------------------------------------
    context_parts = []

    for i, row in results.iterrows():

        context_parts.append(
            f"[Source {i + 1}]\n"
            f"Document ID: {row['document_id']}\n"
            f"Document Type: {row['label']}\n"
            f"Chunk: {row['chunk_index']}\n"
            f"Content:\n{row['text']}"
        )

    context = "\n\n".join(context_parts)

    # ---------------------------------------------
    # Strict grounding prompt
    # ---------------------------------------------
    system_prompt = """
You are DocuMind, an enterprise document assistant.

Answer the user's question ONLY from the supplied document context.

Strict rules:
1. Do not use outside knowledge.
2. Do not invent clauses, fees, amounts, dates, or facts.
3. Only mention a document when the supplied context directly supports
   the claim you make about that document.
4. If the context does not provide enough evidence, say:
   "The retrieved documents do not provide enough evidence."
5. Cite the document ID after each important claim.
6. Do not claim that a document contains a specific clause unless that
   clause is actually present in the supplied context.
"""

    user_prompt = (
        f"DOCUMENT CONTEXT:\n\n"
        f"{context}\n\n"
        f"USER QUESTION:\n{query}"
    )

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    # ---------------------------------------------
    # Generate answer
    # ---------------------------------------------
    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer, results


print("Final full-text RAG generator ready")

Final full-text RAG generator ready


In [5]:
query = "Which contracts mention early termination penalties?"

answer, retrieved_results = generate_full_rag_answer(
    query,
    top_k=5,
    label_filter="Contract"
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nSOURCES USED:")
for _, row in retrieved_results.iterrows():
    print(
        f"- {row['document_id']} "
        f"(chunk {row['chunk_index']})"
    )

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

NameError: name 'llm_tokenizer' is not defined

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

llm_name = "Qwen/Qwen2.5-1.5B-Instruct"

llm_tokenizer = AutoTokenizer.from_pretrained(
    llm_name
)

llm_model = AutoModelForCausalLM.from_pretrained(
    llm_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("LLM loaded")
print("Model:", llm_name)
print("Device:", llm_model.device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LLM loaded
Model: Qwen/Qwen2.5-1.5B-Instruct
Device: cuda:0


In [7]:
query = "Which contracts mention early termination penalties?"

answer, retrieved_results = generate_full_rag_answer(
    query,
    top_k=5,
    label_filter="Contract"
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nSOURCES USED:")
for _, row in retrieved_results.iterrows():
    print(
        f"- {row['document_id']} "
        f"(chunk {row['chunk_index']})"
    )

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION:
Which contracts mention early termination penalties?

ANSWER:
Based on the provided document context, the contracts that mention early termination penalties are:

1. **Contract Number: contract_0112**
   - This contract includes a clause stating that if the other party breaches certain obligations in a material respect, the breaching party must pay an early termination fee equivalent to $1,000,000.

2. **Contract Number: contract_0086**
   - This contract mentions an early termination fee of $1,000,000, which is stated to be equivalent to the amount of outstanding and/or due amounts owed by the company to the repairer under the agreement and/or any other agreement between the repairer and the company.

These contracts contain explicit provisions related to early termination penalties, making them suitable for the user's query.

SOURCES USED:
- contract_0112 (chunk 2)
- contract_0086 (chunk 16)
- contract_0356 (chunk 1)
- contract_0229 (chunk 20)
- contract_0154 (chunk 6)


In [8]:
import re


def get_evidence_chunks(
    query,
    top_k=10,
    candidate_k=50,
    label_filter=None
):
    """
    Retrieve more candidates, then keep only chunks containing
    explicit evidence terms relevant to the question.
    """

    results = hybrid_search_full_rag(
        query,
        top_k=top_k,
        candidate_k=candidate_k,
        label_filter=label_filter
    )

    evidence_terms = [
        "early termination fee",
        "termination fee",
        "termination charge",
        "early termination charge",
        "liquidated damages",
        "termination damages",
        "penalty",
        "termination payment",
        "termination costs"
    ]

    pattern = "|".join(
        re.escape(term)
        for term in evidence_terms
    )

    evidence = results[
        results["text"].str.contains(
            pattern,
            case=False,
            regex=True,
            na=False
        )
    ].copy()

    return evidence.reset_index(drop=True)


def generate_grounded_rag_answer(
    query,
    top_k=10,
    label_filter=None
):

    # ---------------------------------------------
    # Evidence-first retrieval
    # ---------------------------------------------
    evidence_results = get_evidence_chunks(
        query,
        top_k=top_k,
        candidate_k=50,
        label_filter=label_filter
    )

    if len(evidence_results) == 0:

        return (
            "The retrieved documents do not provide "
            "explicit evidence for this question.",
            evidence_results
        )

    # ---------------------------------------------
    # Build evidence-only context
    # ---------------------------------------------
    context_parts = []

    for i, row in evidence_results.iterrows():

        context_parts.append(
            f"[Evidence {i + 1}]\n"
            f"Document ID: {row['document_id']}\n"
            f"Document Type: {row['label']}\n"
            f"Chunk: {row['chunk_index']}\n"
            f"Text:\n{row['text']}"
        )

    context = "\n\n".join(context_parts)

    # ---------------------------------------------
    # Strong anti-hallucination instructions
    # ---------------------------------------------
    system_prompt = """
You are DocuMind, an enterprise document assistant.

Use ONLY the evidence supplied below.

STRICT RULES:

1. Never invent facts.
2. Never infer a monetary amount that is not explicitly present.
3. If an amount is written as [*****], preserve it exactly as [*****].
4. A document qualifies for the user's question only when the supplied
   evidence explicitly supports the claim.
5. A document that merely discusses termination does NOT automatically
   qualify as mentioning a termination fee or penalty.
6. Do not use information from your pretrained knowledge.
7. Do not use information from documents that are not in the evidence.
8. Cite the exact document ID after every claim.
9. When evidence is insufficient, say so instead of guessing.

Return a concise answer.
"""

    user_prompt = (
        f"EVIDENCE:\n\n"
        f"{context}\n\n"
        f"QUESTION:\n{query}"
    )

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer, evidence_results


print("Evidence-grounded RAG generator ready")

Evidence-grounded RAG generator ready


In [9]:
query = "Which contracts mention early termination penalties?"

answer, evidence_results = generate_grounded_rag_answer(
    query,
    top_k=10,
    label_filter="Contract"
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nEVIDENCE SOURCES:")

for _, row in evidence_results.iterrows():

    print(
        f"- {row['document_id']} "
        f"(chunk {row['chunk_index']})"
    )

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION:
Which contracts mention early termination penalties?

ANSWER:
Based on the evidence provided, the contracts that mention early termination penalties are:

1. **Contract ID: contract_0175**
   - Paragraph 14.4.1 specifies the calculation method for liquidated damages in the event of early termination. Specifically, it mentions paying liquidated damages based on the number of approved guest rooms at the hotel.

2. **Contract ID: contract_0086**
   - Paragraph 16.6.1 outlines the consequences of early termination, including the requirement to return certain items and the payment of an early termination fee.

3. **Contract ID: contract_0348**
   - Paragraph 14.4.1 details how liquidated damages would be calculated in the event of early termination, specifying the formula for determining the amount owed.

These clauses clearly indicate that both the BSP (in contract_0175) and the Repairer (in contract_0086) are required to pay liquidated damages or early termination fees in the ev

In [10]:
import json
import re


def generate_quote_grounded_answer(
    query,
    top_k=10,
    label_filter=None
):

    evidence_results = hybrid_search_full_rag(
        query,
        top_k=top_k,
        candidate_k=50,
        label_filter=label_filter
    )

    if len(evidence_results) == 0:
        return (
            "The retrieved documents do not provide enough evidence.",
            evidence_results
        )

    # ---------------------------------------------
    # Build evidence
    # ---------------------------------------------
    context_parts = []

    for i, row in evidence_results.iterrows():

        context_parts.append(
            f"[EVIDENCE {i + 1}]\n"
            f"Document ID: {row['document_id']}\n"
            f"Document Type: {row['label']}\n"
            f"Chunk: {row['chunk_index']}\n"
            f"TEXT:\n{row['text']}"
        )

    context = "\n\n".join(context_parts)

    # ---------------------------------------------
    # Force quote-based output
    # ---------------------------------------------
    system_prompt = """
You are DocuMind, an enterprise document assistant.

Use ONLY the supplied evidence.

For every document you identify:
1. Give the document_id.
2. Give a short explanation.
3. Give an EXACT quote copied from the supplied evidence that supports
   the explanation.

STRICT RULES:
- Never invent facts.
- Never invent quotes.
- Never invent amounts.
- Never infer information that is not explicitly stated.
- If a fee amount is redacted as [*****], write [*****].
- A document only qualifies if the evidence explicitly supports the query.
- Do not include a document merely because it contains the word
  "termination".
- Every claim MUST have an exact supporting quote.

Return ONLY valid JSON in this format:

{
  "results": [
    {
      "document_id": "contract_0086",
      "claim": "The contract explicitly mentions an early termination fee.",
      "quote": "16.5 Early termination fee..."
    }
  ]
}

If there is not enough evidence, return:

{
  "results": []
}
"""

    user_prompt = (
        f"EVIDENCE:\n\n"
        f"{context}\n\n"
        f"QUESTION:\n{query}"
    )

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    raw_answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # ---------------------------------------------
    # Parse JSON
    # ---------------------------------------------
    try:
        parsed = json.loads(raw_answer)

    except json.JSONDecodeError:
        return (
            "The model did not return valid grounded evidence.",
            evidence_results
        )

    # ---------------------------------------------
    # Verify document IDs + exact quotes
    # ---------------------------------------------
    verified_results = []

    for item in parsed.get("results", []):

        document_id = item.get("document_id")
        claim = item.get("claim", "")
        quote = item.get("quote", "")

        matching_rows = evidence_results[
            evidence_results["document_id"] == document_id
        ]

        if len(matching_rows) == 0:
            continue

        quote_found = False

        for _, row in matching_rows.iterrows():

            source_text = str(row["text"])

            if quote.strip() in source_text:
                quote_found = True
                break

        if quote_found:

            verified_results.append({
                "document_id": document_id,
                "claim": claim,
                "quote": quote
            })

    # ---------------------------------------------
    # Build final answer
    # ---------------------------------------------
    if not verified_results:

        answer = (
            "The retrieved documents do not provide enough "
            "verified evidence to answer the question."
        )

    else:

        lines = [
            "Based on the retrieved evidence:"
        ]

        for item in verified_results:

            lines.append(
                f"\n{item['document_id']}: "
                f"{item['claim']}"
            )

            lines.append(
                f"Evidence: \"{item['quote']}\""
            )

        answer = "\n".join(lines)

    return answer, evidence_results


print("Quote-grounded RAG ready")

Quote-grounded RAG ready


In [11]:
query = "Which contracts mention early termination penalties?"

answer, evidence_results = generate_quote_grounded_answer(
    query,
    top_k=10,
    label_filter="Contract"
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nEVIDENCE DOCUMENTS:")

for _, row in evidence_results.iterrows():
    print(
        f"- {row['document_id']} "
        f"(chunk {row['chunk_index']})"
    )

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION:
Which contracts mention early termination penalties?

ANSWER:
The model did not return valid grounded evidence.

EVIDENCE DOCUMENTS:
- contract_0175 (chunk 12)
- contract_0112 (chunk 2)
- contract_0086 (chunk 16)
- contract_0348 (chunk 0)
- contract_0356 (chunk 1)
- contract_0229 (chunk 20)
- contract_0180 (chunk 3)
- contract_0154 (chunk 6)
- contract_0138 (chunk 32)
- contract_0359 (chunk 7)


In [12]:
import json
import re


def extract_json_object(text):
    """
    Extract a JSON object even if the model wraps it in
    ```json ... ``` or adds extra text.
    """

    text = text.strip()

    # Remove markdown code fences
    text = re.sub(
        r"^```(?:json)?\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r"\s*```$",
        "",
        text
    )

    # Try direct JSON parsing
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try extracting the first {...} block
    match = re.search(
        r"\{.*\}",
        text,
        flags=re.DOTALL
    )

    if match:
        candidate = match.group(0)

        try:
            return json.loads(candidate)
        except json.JSONDecodeError:
            pass

    return None


def generate_quote_grounded_answer(
    query,
    top_k=8,
    label_filter=None
):

    # ---------------------------------------------
    # Retrieve evidence
    # ---------------------------------------------
    evidence_results = hybrid_search_full_rag(
        query,
        top_k=top_k,
        candidate_k=50,
        label_filter=label_filter
    )

    if len(evidence_results) == 0:

        return (
            "The retrieved documents do not provide enough evidence.",
            evidence_results
        )

    # ---------------------------------------------
    # Build compact evidence context
    # ---------------------------------------------
    context_parts = []

    for i, row in evidence_results.iterrows():

        context_parts.append(
            f"[EVIDENCE {i + 1}]\n"
            f"Document ID: {row['document_id']}\n"
            f"Chunk: {row['chunk_index']}\n"
            f"{row['text']}"
        )

    context = "\n\n".join(context_parts)

    # ---------------------------------------------
    # Simpler JSON instructions
    # ---------------------------------------------
    system_prompt = """
You are DocuMind.

Answer ONLY from the supplied evidence.

For each qualifying document return:
- document_id
- claim
- quote

Rules:
- The quote MUST be copied exactly from the evidence.
- Do not invent facts.
- Do not invent numbers.
- Do not infer a penalty from ordinary termination language.
- A document qualifies only if the evidence directly supports the question.
- If a fee amount is [*****], preserve [*****].
- Return ONLY JSON.
- No markdown.
- No ``` fences.

Required format:

{
  "results": [
    {
      "document_id": "contract_0086",
      "claim": "The contract explicitly mentions an early termination fee.",
      "quote": "Early termination fee"
    }
  ]
}

If no evidence supports the question:

{
  "results": []
}
"""

    user_prompt = (
        f"EVIDENCE:\n\n"
        f"{context}\n\n"
        f"QUESTION:\n{query}"
    )

    messages = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = llm_tokenizer(
        prompt,
        return_tensors="pt"
    ).to(llm_model.device)

    with torch.no_grad():

        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=250,
            do_sample=False
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    raw_answer = llm_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # ---------------------------------------------
    # Parse model output
    # ---------------------------------------------
    parsed = extract_json_object(
        raw_answer
    )

    # If JSON parsing fails, return diagnostic
    if parsed is None:

        return (
            "The model returned invalid structured output.\n\n"
            "RAW MODEL OUTPUT:\n"
            + raw_answer,
            evidence_results
        )

    # ---------------------------------------------
    # Verify every quote against retrieved evidence
    # ---------------------------------------------
    verified_results = []

    for item in parsed.get("results", []):

        document_id = item.get(
            "document_id",
            ""
        )

        claim = item.get(
            "claim",
            ""
        )

        quote = item.get(
            "quote",
            ""
        )

        matching_rows = evidence_results[
            evidence_results["document_id"]
            == document_id
        ]

        if len(matching_rows) == 0:
            continue

        # Exact quote verification
        quote_found = False

        for _, row in matching_rows.iterrows():

            source_text = str(
                row["text"]
            )

            if quote.strip() in source_text:

                quote_found = True
                break

        if quote_found:

            verified_results.append({
                "document_id": document_id,
                "claim": claim,
                "quote": quote
            })

    # ---------------------------------------------
    # Final grounded answer
    # ---------------------------------------------
    if not verified_results:

        answer = (
            "The retrieved documents do not provide "
            "enough verified evidence to answer the question."
        )

    else:

        answer_lines = [
            "Based on the retrieved evidence:"
        ]

        for item in verified_results:

            answer_lines.append(
                f"\n{item['document_id']}: "
                f"{item['claim']}"
            )

            answer_lines.append(
                f'Evidence: "{item["quote"]}"'
            )

        answer = "\n".join(
            answer_lines
        )

    return answer, evidence_results


print("Robust quote-grounded RAG ready")

Robust quote-grounded RAG ready


In [13]:
query = "Which contracts mention early termination penalties?"

answer, evidence_results = generate_quote_grounded_answer(
    query,
    top_k=8,
    label_filter="Contract"
)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

print("\nEVIDENCE SOURCES:")

for _, row in evidence_results.iterrows():
    print(
        f"- {row['document_id']} "
        f"(chunk {row['chunk_index']})"
    )

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION:
Which contracts mention early termination penalties?

ANSWER:
The model returned invalid structured output.

RAW MODEL OUTPUT:
Based on the evidence provided, the following contracts mention early termination penalties:

1. **Network Management Outsourcing Agreement**  
   - Paragraph 14.4: "Liquidated Damages on Termination."

2. **Technology Outsourcement Agreement**  
   - Paragraph 14.4: "Liquidated Damages on Termination."

These clauses indicate that the contracting parties agree to pay a specific amount (liquidated damages) upon the early termination of the agreement.

EVIDENCE SOURCES:
- contract_0175 (chunk 12)
- contract_0112 (chunk 2)
- contract_0086 (chunk 16)
- contract_0348 (chunk 0)
- contract_0356 (chunk 1)
- contract_0229 (chunk 20)
- contract_0180 (chunk 3)
- contract_0154 (chunk 6)


In [14]:
import re


def extract_verified_evidence(
    query,
    label_filter=None,
    top_k=15
):
    """
    Retrieve chunks and keep only sentences that contain:
    - termination language
    - AND a penalty/fee/damages concept

    This prevents documents that merely mention termination
    from being treated as penalty evidence.
    """

    results = hybrid_search_full_rag(
        query,
        top_k=top_k,
        candidate_k=50,
        label_filter=label_filter
    )

    termination_pattern = re.compile(
        r"\btermination\b|\bterminate\b|\bterminated\b",
        re.IGNORECASE
    )

    penalty_pattern = re.compile(
        r"early\s+termination\s+fee|"
        r"termination\s+fee|"
        r"early\s+termination\s+charge|"
        r"termination\s+charge|"
        r"liquidated\s+damages|"
        r"termination\s+damages|"
        r"termination\s+penalt(?:y|ies)|"
        r"penalt(?:y|ies)|"
        r"termination\s+costs",
        re.IGNORECASE
    )

    evidence_rows = []

    for _, row in results.iterrows():

        text = str(row["text"])

        # Rough sentence splitting
        sentences = re.split(
            r"(?<=[.!?])\s+",
            text
        )

        for sentence in sentences:

            has_termination = bool(
                termination_pattern.search(sentence)
            )

            has_penalty = bool(
                penalty_pattern.search(sentence)
            )

            if has_termination and has_penalty:

                evidence_rows.append({
                    "document_id": row["document_id"],
                    "label": row["label"],
                    "chunk_index": row["chunk_index"],
                    "rrf_score": row["rrf_score"],
                    "evidence": sentence.strip()
                })

    evidence_df = pd.DataFrame(
        evidence_rows
    )

    if len(evidence_df) == 0:
        return evidence_df

    # Remove duplicate evidence
    evidence_df = evidence_df.drop_duplicates(
        subset=[
            "document_id",
            "evidence"
        ]
    )

    # Strongest retrieval result first
    evidence_df = evidence_df.sort_values(
        "rrf_score",
        ascending=False
    ).reset_index(drop=True)

    return evidence_df


print("Verified evidence extractor ready")

Verified evidence extractor ready


In [15]:
query = "Which contracts mention early termination penalties?"

verified = extract_verified_evidence(
    query,
    label_filter="Contract",
    top_k=15
)

print(
    "Verified evidence rows:",
    len(verified)
)

display(
    verified[
        [
            "document_id",
            "chunk_index",
            "evidence"
        ]
    ]
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Verified evidence rows: 22


,document_id,chunk_index,evidence
0,contract_0175,12,14.5 Termination Fee If the BSP terminates und...
1,contract_0175,12,14.6 Sole Remedy The amount payable by the BSP...
2,contract_0086,16,16.5 Early termination fee: subject to not bei...
3,contract_0086,16,"Notwithstanding this Clause 16 (""Termination"")..."
4,contract_0348,0,"As of the Effective Date, the parties acknowle..."
5,contract_0229,20,23 14.4 Liquidated Damages on Termination.
6,contract_0229,20,If this Agreement terminates before the Expira...
7,contract_0229,20,14.4.1.2 If termination occurs after you begin...
8,contract_0229,20,14.4.1.3 If termination occurs after the Effec...
9,contract_0229,20,14.4.1.4 If termination occurs after the secon...


In [16]:
# Group verified evidence by document.
# We keep the strongest retrieval score and all distinct evidence snippets.

verified_contracts = (
    verified
    .groupby("document_id")
    .agg(
        best_rrf_score=("rrf_score", "max"),
        evidence_count=("evidence", "count"),
        evidence=("evidence", lambda x: list(dict.fromkeys(x)))
    )
    .reset_index()
    .sort_values(
        "best_rrf_score",
        ascending=False
    )
    .reset_index(drop=True)
)

print(
    "Contracts with verified termination-related evidence:",
    len(verified_contracts)
)

for _, row in verified_contracts.iterrows():

    print("\n" + "=" * 100)
    print("DOCUMENT:", row["document_id"])
    print("EVIDENCE COUNT:", row["evidence_count"])

    for evidence in row["evidence"]:
        print("\n-", evidence)

Contracts with verified termination-related evidence: 6

DOCUMENT: contract_0175
EVIDENCE COUNT: 2

- 14.5 Termination Fee If the BSP terminates under clause 14.3 (Termination by BSP for Convenience) the BSP will pay Datec the Termination Fee calculated in accordance with Attachment C (Pricing).

- 14.6 Sole Remedy The amount payable by the BSP under clause 14.3 (Termination Fee) will be Datec's sole remedy for the BSP terminating for convenience.

DOCUMENT: contract_0086
EVIDENCE COUNT: 2

- 16.5 Early termination fee: subject to not being in breach of any of its obligation under the Agreement, the Company may terminate this Agreement for convenience by way of Notice of termination; the Agreement shall be then terminated following a [*****] period as from [*****] Confidential material redacted and filed separately with the Securities and Exchange Commission.

- Notwithstanding this Clause 16 ("Termination"), upon receipt of such Notice of termination and without prejudice to any right

In [17]:
rag_context = []

for _, row in verified_contracts.iterrows():
    evidence_text = "\n".join(
        f"- {evidence}"
        for evidence in row["evidence"]
    )

    rag_context.append(
        f"DOCUMENT: {row['document_id']}\n"
        f"EVIDENCE:\n{evidence_text}"
    )

rag_context_text = "\n\n".join(rag_context)

print(rag_context_text)

DOCUMENT: contract_0175
EVIDENCE:
- 14.5 Termination Fee If the BSP terminates under clause 14.3 (Termination by BSP for Convenience) the BSP will pay Datec the Termination Fee calculated in accordance with Attachment C (Pricing).
- 14.6 Sole Remedy The amount payable by the BSP under clause 14.3 (Termination Fee) will be Datec's sole remedy for the BSP terminating for convenience.

DOCUMENT: contract_0086
EVIDENCE:
- 16.5 Early termination fee: subject to not being in breach of any of its obligation under the Agreement, the Company may terminate this Agreement for convenience by way of Notice of termination; the Agreement shall be then terminated following a [*****] period as from [*****] Confidential material redacted and filed separately with the Securities and Exchange Commission.
- Notwithstanding this Clause 16 ("Termination"), upon receipt of such Notice of termination and without prejudice to any rights it may have at Law, the Repairer shall invoice to the Company an early term

In [20]:
prompt = f"""
You are an enterprise document QA assistant.

Answer the user's question using ONLY the verified evidence below.

Rules:
1. Do not invent facts, amounts, dates, or contract details.
2. Do not infer that "liquidated damages" means "penalty".
3. Distinguish between:
   - termination fee
   - early termination fee
   - termination charge
   - liquidated damages
4. If an amount is REDACTED, explicitly say it is redacted.
5. Mention the contract_id for every claim.
6. If the evidence does not establish something, say that it is not established by the retrieved evidence.

Question:
Which contracts mention early termination penalties or fees?

Verified evidence:
{rag_context_text}

Provide a concise answer with:
- contract ID
- type of termination-related payment
- what the evidence says
"""

messages = [
    {
        "role": "system",
        "content": "You answer strictly from provided enterprise document evidence."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.1,
        do_sample=False
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer)

NameError: name 'model' is not defined

In [19]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded:", type(tokenizer).__name__)

Tokenizer loaded: Qwen2Tokenizer


In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

print("Model loaded successfully")
print("Device:", model.device)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


Model loaded successfully
Device: cuda:0


In [22]:
prompt = f"""
You are an enterprise document QA assistant.

Answer the user's question using ONLY the verified evidence below.

Rules:
1. Do not invent facts, amounts, dates, or contract details.
2. Do not infer that "liquidated damages" means "penalty".
3. Distinguish between:
   - termination fee
   - early termination fee
   - termination charge
   - liquidated damages
4. If an amount is REDACTED, explicitly say it is redacted.
5. Mention the contract_id for every claim.
6. If the evidence does not establish something, say that it is not established by the retrieved evidence.

Question:
Which contracts mention early termination penalties or fees?

Verified evidence:
{rag_context_text}

Provide a concise answer with:
- contract ID
- type of termination-related payment
- what the evidence says
"""

messages = [
    {
        "role": "system",
        "content": "You answer strictly from provided enterprise document evidence."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=500,
        temperature=0.1,
        do_sample=False
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


**Contract ID:** contract_0086  
**Type of termination-related payment:** Early termination fee  

**Evidence:**  
- **16.5 Early termination fee:** Subject to not being in breach of any of its obligation under the Agreement, the Company may terminate this Agreement for convenience by way of Notice of termination; the Agreement shall be then terminated following a [*****] period as from [*****]. Confidential material redacted and filed separately with the Securities and Exchange Commission.  
- **Notwithstanding this Clause 16 ("Termination"), upon receipt of such Notice of termination and without prejudice to any rights it may have at Law, the Repairer shall invoice to the Company an early termination fee equivalent to [*****], which shall be paid within [*****] as from the issuance date of the said invoice and/or set off against any outstanding or due payment to the Company, at the Repairer's discretion.  

**Conclusion:** The evidence indicates that contract_0086 mentions an early t

In [23]:
for _, row in verified_contracts.iterrows():

    print(f"\nContract: {row['document_id']}")

    for evidence in row["evidence"]:
        evidence_lower = evidence.lower()

        if "early termination fee" in evidence_lower:
            payment_type = "Early termination fee"
        elif "termination fee" in evidence_lower:
            payment_type = "Termination fee"
        elif "liquidated damages" in evidence_lower:
            payment_type = "Liquidated damages"
        elif "termination charges" in evidence_lower:
            payment_type = "Termination charges"
        else:
            payment_type = "Termination-related payment"

        print(f"Type: {payment_type}")
        print(f"Evidence: {evidence}")


Contract: contract_0175
Type: Termination fee
Evidence: 14.5 Termination Fee If the BSP terminates under clause 14.3 (Termination by BSP for Convenience) the BSP will pay Datec the Termination Fee calculated in accordance with Attachment C (Pricing).
Type: Termination fee
Evidence: 14.6 Sole Remedy The amount payable by the BSP under clause 14.3 (Termination Fee) will be Datec's sole remedy for the BSP terminating for convenience.

Contract: contract_0086
Type: Early termination fee
Evidence: 16.5 Early termination fee: subject to not being in breach of any of its obligation under the Agreement, the Company may terminate this Agreement for convenience by way of Notice of termination; the Agreement shall be then terminated following a [*****] period as from [*****] Confidential material redacted and filed separately with the Securities and Exchange Commission.
Type: Early termination fee
Evidence: Notwithstanding this Clause 16 ("Termination"), upon receipt of such Notice of terminatio

In [24]:
contract_summary = []

for _, row in verified_contracts.iterrows():

    doc_id = row["document_id"]
    evidence_list = row["evidence"]

    types = []

    for evidence in evidence_list:
        e = evidence.lower()

        if "early termination fee" in e:
            types.append("Early termination fee")
        elif "liquidated damages" in e:
            types.append("Liquidated damages")
        elif "termination charges" in e:
            types.append("Termination charges")
        elif "termination fee" in e:
            types.append("Termination fee")

    # Remove duplicate types while preserving order
    types = list(dict.fromkeys(types))

    contract_summary.append({
        "document_id": doc_id,
        "payment_types": types,
        "evidence": evidence_list
    })

for item in contract_summary:
    print("=" * 80)
    print("DOCUMENT:", item["document_id"])
    print("TYPE:", ", ".join(item["payment_types"]))

    for evidence in item["evidence"]:
        print("-", evidence)

DOCUMENT: contract_0175
TYPE: Termination fee
- 14.5 Termination Fee If the BSP terminates under clause 14.3 (Termination by BSP for Convenience) the BSP will pay Datec the Termination Fee calculated in accordance with Attachment C (Pricing).
- 14.6 Sole Remedy The amount payable by the BSP under clause 14.3 (Termination Fee) will be Datec's sole remedy for the BSP terminating for convenience.
DOCUMENT: contract_0086
TYPE: Early termination fee
- 16.5 Early termination fee: subject to not being in breach of any of its obligation under the Agreement, the Company may terminate this Agreement for convenience by way of Notice of termination; the Agreement shall be then terminated following a [*****] period as from [*****] Confidential material redacted and filed separately with the Securities and Exchange Commission.
- Notwithstanding this Clause 16 ("Termination"), upon receipt of such Notice of termination and without prejudice to any rights it may have at Law, the Repairer shall invoice

In [25]:
print("Grounded Answer\n")
print("The verified evidence identifies the following contracts with termination-related payment provisions:\n")

for item in contract_summary:
    doc_id = item["document_id"]

    if doc_id == "contract_0175":
        print(
            "• contract_0175 — Termination Fee: "
            "BSP pays Datec a Termination Fee if BSP terminates for convenience."
        )

    elif doc_id == "contract_0086":
        print(
            "• contract_0086 — Early termination fee: "
            "the Repairer invoices the Company an early termination fee; "
            "the amount is redacted."
        )

    elif doc_id == "contract_0348":
        print(
            "• contract_0348 — Termination Fee: "
            "Customer must pay a Termination Fee for termination for convenience, "
            "calculated using the Estimated Remaining Value."
        )

    elif doc_id == "contract_0229":
        print(
            "• contract_0229 — Liquidated Damages on Termination: "
            "the agreement specifies several formulas for calculating damages "
            "depending on when termination occurs."
        )

    elif doc_id == "contract_0138":
        print(
            "• contract_0138 — Termination Charges: "
            "the cited clause states that AT&T is not liable for termination "
            "charges when the specified contractual termination right is exercised."
        )

    elif doc_id == "contract_0400":
        print(
            "• contract_0400 — Early Termination Fee / Termination for Cause Fee: "
            "both amounts are based on the Estimated Remaining Value and are redacted."
        )

Grounded Answer

The verified evidence identifies the following contracts with termination-related payment provisions:

• contract_0175 — Termination Fee: BSP pays Datec a Termination Fee if BSP terminates for convenience.
• contract_0086 — Early termination fee: the Repairer invoices the Company an early termination fee; the amount is redacted.
• contract_0348 — Termination Fee: Customer must pay a Termination Fee for termination for convenience, calculated using the Estimated Remaining Value.
• contract_0229 — Liquidated Damages on Termination: the agreement specifies several formulas for calculating damages depending on when termination occurs.
• contract_0138 — Termination Charges: the cited clause states that AT&T is not liable for termination charges when the specified contractual termination right is exercised.
• contract_0400 — Early Termination Fee / Termination for Cause Fee: both amounts are based on the Estimated Remaining Value and are redacted.


In [26]:
def answer_question(question):
    """
    Answer an enterprise document question using:
    1. Hybrid retrieval
    2. Evidence verification
    3. Deterministic fact extraction
    4. Optional LLM summarization
    """

    print("=" * 100)
    print("QUESTION")
    print(question)
    print("=" * 100)

    # --------------------------------------------------
    # 1. Retrieve relevant chunks
    # --------------------------------------------------
    results = hybrid_search_full_rag(
        question,
        top_k=50,
        candidate_k=30
    )

    print("\nRetrieved chunks:", len(results))

    # --------------------------------------------------
    # 2. Evidence verification
    # --------------------------------------------------
    verified = verify_termination_evidence(results)

    if len(verified) == 0:
        return "No verified termination-related evidence was found."

    # --------------------------------------------------
    # 3. Group evidence by document
    # --------------------------------------------------
    verified_contracts = (
        verified
        .groupby("document_id")
        .agg(
            best_rrf_score=("rrf_score", "max"),
            evidence_count=("evidence", "count"),
            evidence=("evidence", lambda x: list(dict.fromkeys(x)))
        )
        .reset_index()
        .sort_values(
            "best_rrf_score",
            ascending=False
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------
    # 4. Deterministic answer
    # --------------------------------------------------
    answer_lines = []

    for _, row in verified_contracts.iterrows():

        doc_id = row["document_id"]
        evidence_list = row["evidence"]

        types = []

        for evidence in evidence_list:

            e = evidence.lower()

            if "early termination fee" in e:
                payment_type = "Early termination fee"

            elif "liquidated damages" in e:
                payment_type = "Liquidated damages"

            elif "termination charges" in e:
                payment_type = "Termination charges"

            elif "termination fee" in e:
                payment_type = "Termination fee"

            else:
                payment_type = "Termination-related payment"

            if payment_type not in types:
                types.append(payment_type)

        answer_lines.append(
            f"{doc_id} — {', '.join(types)}"
        )

    print("\nGROUNDED ANSWER\n")

    for line in answer_lines:
        print("•", line)

    print("\nVerified documents:", len(verified_contracts))

    return verified_contracts

In [27]:
import re

termination_pattern = re.compile(
    r"\btermination\b|\bterminate\b|\bterminated\b",
    re.IGNORECASE
)

payment_pattern = re.compile(
    r"""
    early\s+termination\s+fee
    |termination\s+fee
    |early\s+termination\s+charge
    |termination\s+charge
    |liquidated\s+damages
    |termination\s+damages
    |termination\s+penalt(?:y|ies)
    |penalt(?:y|ies)
    |termination\s+costs
    """,
    re.IGNORECASE | re.VERBOSE
)


def verify_termination_evidence(results):
    """
    Keep retrieved chunks that contain both:
    1. termination-related language
    2. payment/penalty-related language
    """

    verified_rows = []

    for _, row in results.iterrows():

        text = str(row["text"])

        if not termination_pattern.search(text):
            continue

        if not payment_pattern.search(text):
            continue

        # Extract sentences containing both concepts
        sentences = re.split(r'(?<=[.!?])\s+', text)

        for sentence in sentences:

            has_termination = termination_pattern.search(sentence)
            has_payment = payment_pattern.search(sentence)

            if has_termination and has_payment:

                verified_rows.append({
                    "document_id": row["document_id"],
                    "chunk_id": row["chunk_id"],
                    "rrf_score": row["rrf_score"],
                    "evidence": sentence.strip()
                })

    return pd.DataFrame(verified_rows)

In [28]:
import re

termination_pattern = re.compile(
    r"\btermination\b|\bterminate\b|\bterminated\b",
    re.IGNORECASE
)

payment_pattern = re.compile(
    r"""
    early\s+termination\s+fee
    |termination\s+fee
    |early\s+termination\s+charge
    |termination\s+charge
    |liquidated\s+damages
    |termination\s+damages
    |termination\s+penalt(?:y|ies)
    |penalt(?:y|ies)
    |termination\s+costs
    """,
    re.IGNORECASE | re.VERBOSE
)


def verify_termination_evidence(results):
    """
    Keep retrieved chunks that contain both:
    1. termination-related language
    2. payment/penalty-related language
    """

    verified_rows = []

    for _, row in results.iterrows():

        text = str(row["text"])

        if not termination_pattern.search(text):
            continue

        if not payment_pattern.search(text):
            continue

        # Extract sentences containing both concepts
        sentences = re.split(r'(?<=[.!?])\s+', text)

        for sentence in sentences:

            has_termination = termination_pattern.search(sentence)
            has_payment = payment_pattern.search(sentence)

            if has_termination and has_payment:

                verified_rows.append({
                    "document_id": row["document_id"],
                    "chunk_id": row["chunk_id"],
                    "rrf_score": row["rrf_score"],
                    "evidence": sentence.strip()
                })

    return pd.DataFrame(verified_rows)

In [29]:
test_results = hybrid_search_full_rag(
    "Which contracts mention early termination penalties or fees?",
    top_k=50,
    candidate_k=30
)

verified_test = verify_termination_evidence(test_results)

print("Verified evidence rows:", len(verified_test))
print("Documents:", verified_test["document_id"].nunique())

print(
    verified_test["document_id"]
    .drop_duplicates()
    .tolist()
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Verified evidence rows: 80
Documents: 38
['report_1862', 'report_1656', 'report_1860', 'contract_0086', 'report_1861', 'report_1843', 'report_1859', 'report_1488', 'report_1487', 'report_1590', 'report_1485', 'contract_0229', 'report_1490', 'report_0692', 'report_1489', 'report_0888', 'report_1486', 'report_1430', 'report_1697', 'report_0568', 'report_0128', 'report_0215', 'report_0007', 'report_0217', 'report_0085', 'report_0216', 'report_1401', 'report_1752', 'report_1404', 'report_0213', 'report_0092', 'report_0566', 'contract_0400', 'report_1858', 'report_0070', 'report_1748', 'report_1746', 'report_1842']


In [30]:
test_results = hybrid_search_full_rag(
    "Which contracts mention early termination penalties or fees?",
    top_k=50,
    candidate_k=30,
    label_filter="Contract"
)

verified_test = verify_termination_evidence(test_results)

print("Verified evidence rows:", len(verified_test))
print("Documents:", verified_test["document_id"].nunique())

print(
    verified_test["document_id"]
    .drop_duplicates()
    .tolist()
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Verified evidence rows: 34
Documents: 11
['contract_0086', 'contract_0229', 'contract_0400', 'contract_0348', 'contract_0408', 'contract_0138', 'contract_0385', 'contract_0175', 'contract_0030', 'contract_0479', 'contract_0266']


In [31]:
for doc_id in verified_test["document_id"].drop_duplicates():

    print("\n" + "=" * 100)
    print("DOCUMENT:", doc_id)

    doc_rows = verified_test[
        verified_test["document_id"] == doc_id
    ]

    for _, row in doc_rows.iterrows():
        print("\n-", row["evidence"])


DOCUMENT: contract_0086

- 16.5 Early termination fee: subject to not being in breach of any of its obligation under the Agreement, the Company may terminate this Agreement for convenience by way of Notice of termination; the Agreement shall be then terminated following a [*****] period as from [*****] Confidential material redacted and filed separately with the Securities and Exchange Commission.

- Notwithstanding this Clause 16 ("Termination"), upon receipt of such Notice of termination and without prejudice to any rights it may have at Law, the Repairer shall invoice to the Company an early termination fee equivalent to [*****], which shall be paid within [*****] as from the issuance date of the said invoice and/or set off against any outstanding or due payment to the Company, at the Repairer's discretion.

DOCUMENT: contract_0229

- 23 14.4 Liquidated Damages on Termination.

- If this Agreement terminates before the Expiration Date, you will pay us Liquidated Damages as follows:

In [32]:
# Scan all Contract chunks instead of only the top-k retrieved chunks

contract_chunks = full_chunks_df[
    full_chunks_df["label"] == "Contract"
].copy()

print("Contract chunks:", len(contract_chunks))

verified_all_contracts = verify_termination_evidence(
    contract_chunks
)

verified_all_contracts = (
    verified_all_contracts
    .drop_duplicates(
        subset=["document_id", "evidence"]
    )
    .reset_index(drop=True)
)

print(
    "Verified evidence rows:",
    len(verified_all_contracts)
)

print(
    "Distinct contracts:",
    verified_all_contracts["document_id"].nunique()
)

print(
    verified_all_contracts["document_id"]
    .drop_duplicates()
    .tolist()
)

Contract chunks: 6069


KeyError: 'rrf_score'

In [33]:
import re
import pandas as pd

termination_pattern = re.compile(
    r"\btermination\b|\bterminate\b|\bterminated\b",
    re.IGNORECASE
)

payment_pattern = re.compile(
    r"""
    early\s+termination\s+fee
    |termination\s+fee
    |early\s+termination\s+charge
    |termination\s+charge
    |liquidated\s+damages
    |termination\s+damages
    |termination\s+penalt(?:y|ies)
    |penalt(?:y|ies)
    |termination\s+costs
    """,
    re.IGNORECASE | re.VERBOSE
)


def verify_termination_evidence(results):
    """
    Verify chunks containing both termination-related
    language and payment/damages-related language.

    Works with both:
    - hybrid-search results containing rrf_score
    - raw corpus chunks without rrf_score
    """

    verified_rows = []

    for _, row in results.iterrows():

        text = str(row["text"])

        if not termination_pattern.search(text):
            continue

        if not payment_pattern.search(text):
            continue

        sentences = re.split(
            r'(?<=[.!?])\s+',
            text
        )

        for sentence in sentences:

            if not termination_pattern.search(sentence):
                continue

            if not payment_pattern.search(sentence):
                continue

            verified_rows.append({
                "document_id": row["document_id"],
                "chunk_id": row["chunk_id"],
                "rrf_score": row.get("rrf_score", 0.0),
                "evidence": sentence.strip()
            })

    return pd.DataFrame(verified_rows)

In [34]:
contract_chunks = full_chunks_df[
    full_chunks_df["label"] == "Contract"
].copy()

print("Contract chunks:", len(contract_chunks))

verified_all_contracts = verify_termination_evidence(
    contract_chunks
)

verified_all_contracts = (
    verified_all_contracts
    .drop_duplicates(
        subset=["document_id", "evidence"]
    )
    .reset_index(drop=True)
)

print(
    "Verified evidence rows:",
    len(verified_all_contracts)
)

print(
    "Distinct contracts:",
    verified_all_contracts["document_id"].nunique()
)

print(
    verified_all_contracts["document_id"]
    .drop_duplicates()
    .tolist()
)

Contract chunks: 6069
Verified evidence rows: 157
Distinct contracts: 59
['contract_0012', 'contract_0030', 'contract_0034', 'contract_0054', 'contract_0055', 'contract_0086', 'contract_0102', 'contract_0135', 'contract_0138', 'contract_0149', 'contract_0157', 'contract_0160', 'contract_0165', 'contract_0175', 'contract_0177', 'contract_0185', 'contract_0192', 'contract_0212', 'contract_0220', 'contract_0226', 'contract_0229', 'contract_0232', 'contract_0234', 'contract_0235', 'contract_0250', 'contract_0254', 'contract_0256', 'contract_0266', 'contract_0268', 'contract_0290', 'contract_0295', 'contract_0299', 'contract_0302', 'contract_0322', 'contract_0323', 'contract_0324', 'contract_0329', 'contract_0335', 'contract_0348', 'contract_0363', 'contract_0364', 'contract_0385', 'contract_0392', 'contract_0397', 'contract_0400', 'contract_0408', 'contract_0428', 'contract_0432', 'contract_0434', 'contract_0441', 'contract_0442', 'contract_0446', 'contract_0448', 'contract_0449', 'contrac

In [35]:
strict_patterns = {
    "Early termination fee": re.compile(
        r"\bearly\s+termination\s+fee\b",
        re.IGNORECASE
    ),

    "Termination fee": re.compile(
        r"\btermination\s+fee\b",
        re.IGNORECASE
    ),

    "Termination charge": re.compile(
        r"\btermination\s+charges?\b",
        re.IGNORECASE
    ),

    "Liquidated damages": re.compile(
        r"\bliquidated\s+damages\b",
        re.IGNORECASE
    ),

    "Termination penalty": re.compile(
        r"\btermination\s+penalt(?:y|ies)\b",
        re.IGNORECASE
    )
}


def classify_payment_type(evidence):
    matches = []

    for label, pattern in strict_patterns.items():
        if pattern.search(evidence):
            matches.append(label)

    return matches


classified_rows = []

for _, row in verified_all_contracts.iterrows():

    types = classify_payment_type(
        row["evidence"]
    )

    if len(types) > 0:
        classified_rows.append({
            "document_id": row["document_id"],
            "type": ", ".join(types),
            "evidence": row["evidence"]
        })


strict_df = pd.DataFrame(classified_rows)

print("Strict evidence rows:", len(strict_df))
print(
    "Distinct contracts:",
    strict_df["document_id"].nunique()
)

print("\nType distribution:")
print(
    strict_df
    .groupby("type")["document_id"]
    .nunique()
    .sort_values(ascending=False)
)

Strict evidence rows: 109
Distinct contracts: 30

Type distribution:
type
Liquidated damages                         13
Termination fee                            11
Early termination fee, Termination fee      4
Termination charge                          3
Termination penalty                         2
Liquidated damages, Termination penalty     1
Termination fee, Liquidated damages         1
Name: document_id, dtype: int64


In [36]:
strict_summary = (
    strict_df
    .groupby("document_id")
    .agg(
        types=("type", lambda x: ", ".join(sorted(set(x)))),
        evidence=("evidence", lambda x: list(dict.fromkeys(x))[:3])
    )
    .reset_index()
    .sort_values("document_id")
)

print("Contracts:", len(strict_summary))

for _, row in strict_summary.iterrows():

    print("\n" + "=" * 100)
    print("DOCUMENT:", row["document_id"])
    print("TYPE:", row["types"])

    for evidence in row["evidence"]:
        print("-", evidence)

Contracts: 30

DOCUMENT: contract_0030
TYPE: Liquidated damages
- 16.5.1 The payments called for in Section 16.4 [Liquidated Damages] above constitute liquidated damages for causing the premature termination of this Agreement and not a penalty.
- Nevertheless, the parties agree that the lump-sum payment provided under Section 16.4 [Liquidated Damages] above is reasonable in light of the damages for premature termination that may reasonably be expected to
- Nevertheless, the parties agree that the lump-sum payment provided under Section 16.4 [Liquidated Damages] above is reasonable in light of the damages for premature termination that may reasonably be expected to occur in such event.

DOCUMENT: contract_0034
TYPE: Liquidated damages
- 3.2.13 In the cases that Party B guarantees to sign the Financial Leasing Agreement with Party A's users, Party B will agree on the provisions of terminating the Financial Leasing Agreement unilaterally by the Driver User in advance with the Driver User 

In [40]:
def classify_contract_evidence(evidence_list):
    """
    Classify evidence for a contract into:
    - explicit_payment
    - explicit_negative
    - mention_only
    """

    has_payment = False
    has_negative = False

    for evidence in evidence_list:

        e = evidence.lower()

        # Explicit payment obligation
        payment_patterns = [
            r"\bshall pay\b",
            r"\bmust pay\b",
            r"\bwill pay\b",
            r"\bpay to\b",
            r"\bpayable\b",
            r"\bpaid\b",
            r"\binvoice\b",
            r"\breceive\b.*\btermination fee\b",
            r"\bpayment of\b.*\btermination\b",
        ]

        # Explicit negative / exception language
        negative_patterns = [
            r"\bnot\s+.*liable\b",
            r"\bnot\s+.*responsible\b",
            r"\bwithout payment\b",
            r"\bwithout the payment\b",
            r"\bwithout any early termination penalty\b",
            r"\bnot a penalty\b",
            r"\bshall not\b.*\btermination fee\b",
            r"\bshall not\b.*\btermination charges\b",
            r"\bwithout payment of the termination fee\b",
        ]

        for pattern in payment_patterns:
            if re.search(pattern, e, re.IGNORECASE):
                has_payment = True
                break

        for pattern in negative_patterns:
            if re.search(pattern, e, re.IGNORECASE):
                has_negative = True
                break

    if has_payment and has_negative:
        return "payment_with_exception"

    if has_payment:
        return "explicit_payment"

    if has_negative:
        return "explicit_negative"

    return "mention_only"


semantic_rows = []

for doc_id, group in termination_analysis.groupby("document_id"):

    evidence_list = (
        group["evidence"]
        .drop_duplicates()
        .tolist()
    )

    types = (
        group["type"]
        .drop_duplicates()
        .tolist()
    )

    classification = classify_contract_evidence(
        evidence_list
    )

    semantic_rows.append({
        "document_id": doc_id,
        "type": ", ".join(types),
        "classification": classification,
        "evidence": evidence_list
    })


semantic_summary = pd.DataFrame(semantic_rows)

print(
    semantic_summary
    .groupby("classification")["document_id"]
    .nunique()
)

print("\nExplicit payment:")
print(
    semantic_summary[
        semantic_summary["classification"] == "explicit_payment"
    ]["document_id"].tolist()
)

print("\nPayment with exception:")
print(
    semantic_summary[
        semantic_summary["classification"] == "payment_with_exception"
    ]["document_id"].tolist()
)

print("\nExplicit negative:")
print(
    semantic_summary[
        semantic_summary["classification"] == "explicit_negative"
    ]["document_id"].tolist()
)

print("\nMention only:")
print(
    semantic_summary[
        semantic_summary["classification"] == "mention_only"
    ]["document_id"].tolist()
)

classification
explicit_negative          3
explicit_payment          17
mention_only               4
payment_with_exception     6
Name: document_id, dtype: int64

Explicit payment:
['contract_0086', 'contract_0102', 'contract_0135', 'contract_0175', 'contract_0229', 'contract_0232', 'contract_0256', 'contract_0266', 'contract_0302', 'contract_0322', 'contract_0329', 'contract_0335', 'contract_0363', 'contract_0385', 'contract_0446', 'contract_0470', 'contract_0479']

Payment with exception:
['contract_0034', 'contract_0250', 'contract_0348', 'contract_0400', 'contract_0408', 'contract_0441']

Explicit negative:
['contract_0030', 'contract_0138', 'contract_0397']

Mention only:
['contract_0254', 'contract_0290', 'contract_0364', 'contract_0448']


In [38]:
import re
import pandas as pd

# --------------------------------------------------
# 1. Exact terminology patterns
# --------------------------------------------------

strict_patterns = {
    "Early termination fee": re.compile(
        r"\bearly\s+termination\s+fee\b",
        re.IGNORECASE
    ),

    "Termination fee": re.compile(
        r"\btermination\s+fee\b",
        re.IGNORECASE
    ),

    "Termination charge": re.compile(
        r"\btermination\s+charges?\b",
        re.IGNORECASE
    ),

    "Liquidated damages": re.compile(
        r"\bliquidated\s+damages\b",
        re.IGNORECASE
    ),

    "Termination penalty": re.compile(
        r"\btermination\s+penalt(?:y|ies)\b",
        re.IGNORECASE
    )
}


def classify_payment_type(evidence):
    matches = []

    for label, pattern in strict_patterns.items():
        if pattern.search(evidence):
            matches.append(label)

    return matches


# --------------------------------------------------
# 2. Payment / negative language
# --------------------------------------------------

payment_patterns = [
    r"\bshall pay\b",
    r"\bmust pay\b",
    r"\bwill pay\b",
    r"\bpay to\b",
    r"\bpayable\b",
    r"\bpaid\b",
    r"\binvoice\b",
    r"\breceive\b.*\btermination fee\b",
    r"\bpayment of\b.*\btermination\b"
]

negative_patterns = [
    r"\bnot\s+.*liable\b",
    r"\bnot\s+.*responsible\b",
    r"\bwithout payment\b",
    r"\bwithout the payment\b",
    r"\bwithout any early termination penalty\b",
    r"\bnot a penalty\b",
    r"\bshall not\b.*\btermination fee\b",
    r"\bshall not\b.*\btermination charges\b",
    r"\bwithout payment of the termination fee\b"
]


def classify_contract_evidence(evidence_list):

    has_payment = False
    has_negative = False

    for evidence in evidence_list:

        e = str(evidence)

        for pattern in payment_patterns:
            if re.search(pattern, e, re.IGNORECASE):
                has_payment = True
                break

        for pattern in negative_patterns:
            if re.search(pattern, e, re.IGNORECASE):
                has_negative = True
                break

    if has_payment and has_negative:
        return "payment_with_exception"

    if has_payment:
        return "explicit_payment"

    if has_negative:
        return "explicit_negative"

    return "mention_only"


# --------------------------------------------------
# 3. Rebuild termination_analysis
# --------------------------------------------------

classified_rows = []

for _, row in verified_all_contracts.iterrows():

    evidence = row["evidence"]

    types = classify_payment_type(evidence)

    if len(types) == 0:
        continue

    classified_rows.append({
        "document_id": row["document_id"],
        "type": ", ".join(types),
        "evidence": evidence
    })


termination_analysis = pd.DataFrame(classified_rows)


# --------------------------------------------------
# 4. Group by contract
# --------------------------------------------------

semantic_rows = []

for doc_id, group in termination_analysis.groupby("document_id"):

    evidence_list = (
        group["evidence"]
        .drop_duplicates()
        .tolist()
    )

    types = (
        group["type"]
        .drop_duplicates()
        .tolist()
    )

    classification = classify_contract_evidence(
        evidence_list
    )

    semantic_rows.append({
        "document_id": doc_id,
        "type": ", ".join(types),
        "classification": classification,
        "evidence": evidence_list
    })


semantic_summary = pd.DataFrame(semantic_rows)


# --------------------------------------------------
# 5. Results
# --------------------------------------------------

print("Total contracts with strict terminology:",
      len(semantic_summary))

print("\nClassification counts:")
print(
    semantic_summary
    .groupby("classification")["document_id"]
    .nunique()
)


print("\n" + "=" * 80)
print("EXPLICIT PAYMENT")
print("=" * 80)

print(
    semantic_summary[
        semantic_summary["classification"] == "explicit_payment"
    ][["document_id", "type"]]
    .to_string(index=False)
)


print("\n" + "=" * 80)
print("PAYMENT WITH EXCEPTION")
print("=" * 80)

print(
    semantic_summary[
        semantic_summary["classification"] == "payment_with_exception"
    ][["document_id", "type"]]
    .to_string(index=False)
)


print("\n" + "=" * 80)
print("EXPLICIT NEGATIVE")
print("=" * 80)

print(
    semantic_summary[
        semantic_summary["classification"] == "explicit_negative"
    ][["document_id", "type"]]
    .to_string(index=False)
)


print("\n" + "=" * 80)
print("MENTION ONLY")
print("=" * 80)

print(
    semantic_summary[
        semantic_summary["classification"] == "mention_only"
    ][["document_id", "type"]]
    .to_string(index=False)
)

Total contracts with strict terminology: 30

Classification counts:
classification
explicit_negative          3
explicit_payment          17
mention_only               4
payment_with_exception     6
Name: document_id, dtype: int64

EXPLICIT PAYMENT
  document_id                                                        type
contract_0086                      Early termination fee, Termination fee
contract_0102                                         Termination penalty
contract_0135                      Early termination fee, Termination fee
contract_0175                                             Termination fee
contract_0229                                          Liquidated damages
contract_0232                                          Liquidated damages
contract_0256                                             Termination fee
contract_0266                                             Termination fee
contract_0302     Early termination fee, Termination fee, Termination fee
contract_03

In [39]:
payment_patterns = [
    r"\bshall pay\b",
    r"\bmust pay\b",
    r"\bwill pay\b",
    r"\bwould pay\b",
    r"\bpay to\b",
    r"\bpay(?:ment)?\s+of\b",
    r"\bpayable\b",
    r"\bpaid\b",
    r"\binvoice\b",
    r"\breceive\b.*\b(?:termination fee|liquidated damages)\b",
    r"\bentitled to receive\b",
    r"\breceive\b.*\bdamages\b",
    r"\brequire\b.*\bto pay\b",
    r"\bagree(?:s)? to pay\b",
    r"\bpayment\b.*\bliquidated damages\b",
    r"\bpayment\b.*\btermination fee\b",
    r"\btermination fee\b.*\bpay\b",
    r"\bliquidated damages\b.*\bpay\b",
    r"\bpay\b.*\bliquidated damages\b",
    r"\bpay\b.*\btermination fee\b"
]

negative_patterns = [
    r"\bnot\s+.*liable\b",
    r"\bnot\s+.*responsible\b",
    r"\bwithout payment\b",
    r"\bwithout the payment\b",
    r"\bwithout any early termination penalty\b",
    r"\bnot a penalty\b",
    r"\bshall not\b.*\btermination fee\b",
    r"\bshall not\b.*\btermination charges\b",
    r"\bwithout payment of the termination fee\b"
]

In [41]:
# Rebuild semantic classification using the UPDATED payment patterns

semantic_rows = []

for doc_id, group in termination_analysis.groupby("document_id"):

    evidence_list = (
        group["evidence"]
        .drop_duplicates()
        .tolist()
    )

    types = sorted(
        set(
            t
            for evidence in evidence_list
            for t in classify_payment_type(evidence)
        )
    )

    classification = classify_contract_evidence(
        evidence_list
    )

    semantic_rows.append({
        "document_id": doc_id,
        "type": ", ".join(types),
        "classification": classification,
        "evidence": evidence_list
    })


semantic_summary = pd.DataFrame(semantic_rows)

print("Classification counts:")
print(
    semantic_summary
    .groupby("classification")["document_id"]
    .nunique()
)

print("\n" + "=" * 80)
print("EXPLICIT PAYMENT")
print("=" * 80)

print(
    semantic_summary[
        semantic_summary["classification"] == "explicit_payment"
    ][["document_id", "type"]]
    .to_string(index=False)
)

print("\n" + "=" * 80)
print("PAYMENT WITH EXCEPTION")
print("=" * 80)

print(
    semantic_summary[
        semantic_summary["classification"] == "payment_with_exception"
    ][["document_id", "type"]]
    .to_string(index=False)
)

print("\n" + "=" * 80)
print("EXPLICIT NEGATIVE")
print("=" * 80)

print(
    semantic_summary[
        semantic_summary["classification"] == "explicit_negative"
    ][["document_id", "type"]]
    .to_string(index=False)
)

print("\n" + "=" * 80)
print("MENTION ONLY")
print("=" * 80)

print(
    semantic_summary[
        semantic_summary["classification"] == "mention_only"
    ][["document_id", "type"]]
    .to_string(index=False)
)

Classification counts:
classification
explicit_negative          3
explicit_payment          17
mention_only               4
payment_with_exception     6
Name: document_id, dtype: int64

EXPLICIT PAYMENT
  document_id                                    type
contract_0086  Early termination fee, Termination fee
contract_0102                     Termination penalty
contract_0135  Early termination fee, Termination fee
contract_0175                         Termination fee
contract_0229                      Liquidated damages
contract_0232                      Liquidated damages
contract_0256                         Termination fee
contract_0266                         Termination fee
contract_0302  Early termination fee, Termination fee
contract_0322                      Liquidated damages
contract_0329                         Termination fee
contract_0335                      Liquidated damages
contract_0363                      Liquidated damages
contract_0385                         Te

In [42]:
# Find the evidence for contract_0364
debug_rows = termination_analysis[
    termination_analysis["document_id"] == "contract_0364"
]

print("contract_0364 evidence:\n")

for _, row in debug_rows.iterrows():

    evidence = str(row["evidence"])

    print("-" * 100)
    print(evidence)

    print("\nPayment matches:")

    for pattern in payment_patterns:
        match = re.search(pattern, evidence, re.IGNORECASE)

        if match:
            print("MATCH:", pattern)
            print("TEXT :", match.group(0))

    print("\nNegative matches:")

    for pattern in negative_patterns:
        match = re.search(pattern, evidence, re.IGNORECASE)

        if match:
            print("MATCH:", pattern)
            print("TEXT :", match.group(0))

    print()

contract_0364 evidence:

----------------------------------------------------------------------------------------------------
In the event that Party B breaches this covenant, Party A has the right to terminate this Agreement at any time and require Party B to pay 30% of Party A's share of revenues as liquidated damages.

Payment matches:
MATCH: \brequire\b.*\bto pay\b
TEXT : require Party B to pay
MATCH: \bpay\b.*\bliquidated damages\b
TEXT : pay 30% of Party A's share of revenues as liquidated damages

Negative matches:



In [43]:
# Direct test of the classifier

evidence_0364 = (
    termination_analysis[
        termination_analysis["document_id"] == "contract_0364"
    ]["evidence"]
    .drop_duplicates()
    .tolist()
)

print("Evidence:")
for e in evidence_0364:
    print(e)

print("\nClassifier result:")
print(classify_contract_evidence(evidence_0364))

Evidence:
In the event that Party B breaches this covenant, Party A has the right to terminate this Agreement at any time and require Party B to pay 30% of Party A's share of revenues as liquidated damages.

Classifier result:
mention_only


In [44]:
def classify_contract_evidence(evidence_list):

    payment_patterns_local = [
        r"\bshall pay\b",
        r"\bmust pay\b",
        r"\bwill pay\b",
        r"\bwould pay\b",
        r"\brequire\b.*\bto pay\b",
        r"\bagree(?:s)? to pay\b",
        r"\bpay to\b",
        r"\bpayable\b",
        r"\bpaid\b",
        r"\binvoice\b",
        r"\bpay\b.*\bliquidated damages\b",
        r"\bpay\b.*\btermination fee\b",
        r"\bpayment of\b.*\btermination\b",
        r"\bpayment of\b.*\bliquidated damages\b",
        r"\bentitled to receive\b",
        r"\breceive\b.*\btermination fee\b",
        r"\breceive\b.*\bdamages\b"
    ]

    negative_patterns_local = [
        r"\bnot\s+.*liable\b",
        r"\bnot\s+.*responsible\b",
        r"\bwithout payment\b",
        r"\bwithout the payment\b",
        r"\bwithout any early termination penalty\b",
        r"\bnot a penalty\b",
        r"\bshall not\b.*\btermination fee\b",
        r"\bshall not\b.*\btermination charges\b",
        r"\bwithout payment of the termination fee\b"
    ]

    has_payment = False
    has_negative = False

    for evidence in evidence_list:

        e = str(evidence)

        for pattern in payment_patterns_local:
            if re.search(pattern, e, re.IGNORECASE):
                has_payment = True
                break

        for pattern in negative_patterns_local:
            if re.search(pattern, e, re.IGNORECASE):
                has_negative = True
                break

    if has_payment and has_negative:
        return "payment_with_exception"

    if has_payment:
        return "explicit_payment"

    if has_negative:
        return "explicit_negative"

    return "mention_only"

In [45]:
evidence_0364 = (
    termination_analysis[
        termination_analysis["document_id"] == "contract_0364"
    ]["evidence"]
    .drop_duplicates()
    .tolist()
)

print(classify_contract_evidence(evidence_0364))

explicit_payment


In [46]:
semantic_rows = []

for doc_id, group in termination_analysis.groupby("document_id"):

    evidence_list = (
        group["evidence"]
        .drop_duplicates()
        .tolist()
    )

    types = sorted(
        set(
            payment_type
            for evidence in evidence_list
            for payment_type in classify_payment_type(evidence)
        )
    )

    classification = classify_contract_evidence(
        evidence_list
    )

    semantic_rows.append({
        "document_id": doc_id,
        "type": ", ".join(types),
        "classification": classification,
        "evidence": evidence_list
    })


semantic_summary = pd.DataFrame(semantic_rows)

print("Classification counts:")
print(
    semantic_summary
    .groupby("classification")["document_id"]
    .nunique()
)

print("\nExplicit payment:")
print(
    semantic_summary[
        semantic_summary["classification"] == "explicit_payment"
    ]["document_id"].tolist()
)

print("\nPayment with exception:")
print(
    semantic_summary[
        semantic_summary["classification"] == "payment_with_exception"
    ]["document_id"].tolist()
)

print("\nExplicit negative:")
print(
    semantic_summary[
        semantic_summary["classification"] == "explicit_negative"
    ]["document_id"].tolist()
)

print("\nMention only:")
print(
    semantic_summary[
        semantic_summary["classification"] == "mention_only"
    ]["document_id"].tolist()
)

Classification counts:
classification
explicit_negative          3
explicit_payment          19
mention_only               2
payment_with_exception     6
Name: document_id, dtype: int64

Explicit payment:
['contract_0086', 'contract_0102', 'contract_0135', 'contract_0175', 'contract_0229', 'contract_0232', 'contract_0256', 'contract_0266', 'contract_0290', 'contract_0302', 'contract_0322', 'contract_0329', 'contract_0335', 'contract_0363', 'contract_0364', 'contract_0385', 'contract_0446', 'contract_0470', 'contract_0479']

Payment with exception:
['contract_0034', 'contract_0250', 'contract_0348', 'contract_0400', 'contract_0408', 'contract_0441']

Explicit negative:
['contract_0030', 'contract_0138', 'contract_0397']

Mention only:
['contract_0254', 'contract_0448']


In [47]:
def grounded_termination_answer():
    """
    Build a grounded answer from the semantic evidence layer.
    No LLM is used to determine which contracts qualify.
    """

    payment_df = semantic_summary[
        semantic_summary["classification"].isin([
            "explicit_payment",
            "payment_with_exception"
        ])
    ].copy()

    negative_df = semantic_summary[
        semantic_summary["classification"] == "explicit_negative"
    ].copy()

    mention_df = semantic_summary[
        semantic_summary["classification"] == "mention_only"
    ].copy()

    print("=" * 100)
    print("GROUNDED CONTRACT QA RESULT")
    print("=" * 100)

    print(
        f"\nContracts with payment-related termination provisions: "
        f"{len(payment_df)}"
    )

    print("\nPayment provisions:\n")

    for _, row in payment_df.sort_values("document_id").iterrows():

        print(
            f"• {row['document_id']} — {row['type']}"
        )

        if row["classification"] == "payment_with_exception":
            print(
                "  Note: the evidence also contains an exception or "
                "condition affecting the payment obligation."
            )

    print(
        f"\nExplicitly negative / non-applicable cases: "
        f"{len(negative_df)}"
    )

    for _, row in negative_df.sort_values("document_id").iterrows():
        print(
            f"• {row['document_id']} — {row['type']}"
        )

    print(
        f"\nTerminology mentioned without established payment obligation: "
        f"{len(mention_df)}"
    )

    for _, row in mention_df.sort_values("document_id").iterrows():
        print(
            f"• {row['document_id']} — {row['type']}"
        )

    return {
        "payment_contracts": payment_df,
        "negative_contracts": negative_df,
        "mention_only_contracts": mention_df
    }


termination_result = grounded_termination_answer()

GROUNDED CONTRACT QA RESULT

Contracts with payment-related termination provisions: 25

Payment provisions:

• contract_0034 — Liquidated damages
  Note: the evidence also contains an exception or condition affecting the payment obligation.
• contract_0086 — Early termination fee, Termination fee
• contract_0102 — Termination penalty
• contract_0135 — Early termination fee, Termination fee
• contract_0175 — Termination fee
• contract_0229 — Liquidated damages
• contract_0232 — Liquidated damages
• contract_0250 — Termination charge
  Note: the evidence also contains an exception or condition affecting the payment obligation.
• contract_0256 — Termination fee
• contract_0266 — Termination fee
• contract_0290 — Liquidated damages
• contract_0302 — Early termination fee, Termination fee
• contract_0322 — Liquidated damages
• contract_0329 — Termination fee
• contract_0335 — Liquidated damages
• contract_0348 — Liquidated damages, Termination fee
  Note: the evidence also contains an excep

In [48]:
def answer_termination_query():

    direct_terms = [
        "Early termination fee",
        "Termination fee",
        "Termination charge",
        "Termination penalty"
    ]

    # Contracts with explicit fee/charge/penalty terminology
    direct = semantic_summary[
        semantic_summary["type"].apply(
            lambda x: any(term in x for term in direct_terms)
        )
    ].copy()

    # Actual payment obligations
    direct_payment = direct[
        direct["classification"].isin([
            "explicit_payment",
            "payment_with_exception"
        ])
    ].copy()

    # Explicitly negative cases
    negative = direct[
        direct["classification"] == "explicit_negative"
    ].copy()

    # Terminology mentioned without payment obligation
    mention_only = direct[
        direct["classification"] == "mention_only"
    ].copy()

    print("=" * 100)
    print("EXACT QUERY: Which contracts mention early termination penalties or fees?")
    print("=" * 100)

    print(
        "\nContracts with explicit fee/charge/penalty terminology "
        "and a payment provision:"
    )

    print(
        direct_payment[
            ["document_id", "type", "classification"]
        ]
        .sort_values("document_id")
        .to_string(index=False)
    )

    print(
        "\nCount:",
        len(direct_payment)
    )

    print("\n" + "=" * 100)
    print("EXPLICIT NEGATIVE / EXCEPTION")
    print("=" * 100)

    print(
        negative[
            ["document_id", "type", "classification"]
        ]
        .sort_values("document_id")
        .to_string(index=False)
    )

    print("\n" + "=" * 100)
    print("TERMINOLOGY ONLY")
    print("=" * 100)

    print(
        mention_only[
            ["document_id", "type", "classification"]
        ]
        .sort_values("document_id")
        .to_string(index=False)
    )

    return {
        "direct_payment": direct_payment,
        "negative": negative,
        "mention_only": mention_only
    }


exact_result = answer_termination_query()

EXACT QUERY: Which contracts mention early termination penalties or fees?

Contracts with explicit fee/charge/penalty terminology and a payment provision:
  document_id                                                       type         classification
contract_0086                     Early termination fee, Termination fee       explicit_payment
contract_0102                                        Termination penalty       explicit_payment
contract_0135                     Early termination fee, Termination fee       explicit_payment
contract_0175                                            Termination fee       explicit_payment
contract_0250                                         Termination charge payment_with_exception
contract_0256                                            Termination fee       explicit_payment
contract_0266                                            Termination fee       explicit_payment
contract_0302                     Early termination fee, Termination fee     

In [49]:
def build_grounded_context(result):

    sections = []

    # Direct payment provisions
    for _, row in result["direct_payment"].sort_values("document_id").iterrows():

        evidence_text = "\n".join(
            f"- {e}"
            for e in row["evidence"]
        )

        sections.append(
            f"DOCUMENT: {row['document_id']}\n"
            f"TYPE: {row['type']}\n"
            f"CLASSIFICATION: {row['classification']}\n"
            f"EVIDENCE:\n{evidence_text}"
        )

    # Explicit negative cases
    for _, row in result["negative"].sort_values("document_id").iterrows():

        evidence_text = "\n".join(
            f"- {e}"
            for e in row["evidence"]
        )

        sections.append(
            f"DOCUMENT: {row['document_id']}\n"
            f"TYPE: {row['type']}\n"
            f"CLASSIFICATION: explicit_negative\n"
            f"EVIDENCE:\n{evidence_text}"
        )

    # Mention only
    for _, row in result["mention_only"].sort_values("document_id").iterrows():

        evidence_text = "\n".join(
            f"- {e}"
            for e in row["evidence"]
        )

        sections.append(
            f"DOCUMENT: {row['document_id']}\n"
            f"TYPE: {row['type']}\n"
            f"CLASSIFICATION: mention_only\n"
            f"EVIDENCE:\n{evidence_text}"
        )

    return "\n\n".join(sections)


grounded_context = build_grounded_context(
    exact_result
)

print(grounded_context)

DOCUMENT: contract_0086
TYPE: Early termination fee, Termination fee
CLASSIFICATION: explicit_payment
EVIDENCE:
- 16.5 Early termination fee: subject to not being in breach of any of its obligation under the Agreement, the Company may terminate this Agreement for convenience by way of Notice of termination; the Agreement shall be then terminated following a [*****] period as from [*****] Confidential material redacted and filed separately with the Securities and Exchange Commission.
- Notwithstanding this Clause 16 ("Termination"), upon receipt of such Notice of termination and without prejudice to any rights it may have at Law, the Repairer shall invoice to the Company an early termination fee equivalent to [*****], which shall be paid within [*****] as from the issuance date of the said invoice and/or set off against any outstanding or due payment to the Company, at the Repairer's discretion.

DOCUMENT: contract_0102
TYPE: Termination penalty
CLASSIFICATION: explicit_payment
EVIDENCE

In [50]:
question = "Which contracts mention early termination penalties or fees?"

prompt = f"""
Answer the question using ONLY the verified evidence below.

Question:
{question}

Rules:
1. Do not invent facts.
2. Do not omit any contract classified as explicit_payment or payment_with_exception.
3. Do not describe liquidated damages as a penalty unless the evidence explicitly does so.
4. Clearly distinguish termination fee, early termination fee,
   termination charge, termination penalty, and liquidated damages.
5. Mention contract IDs.
6. For redacted amounts, say the amount is redacted.
7. Explicitly mention negative cases separately.
8. Mention terminology-only cases separately.
9. Do not use information outside the provided evidence.

Verified evidence:
{grounded_context}
"""

messages = [
    {
        "role": "system",
        "content": (
            "You are an enterprise document QA assistant. "
            "You must answer strictly from verified evidence."
        )
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=12000
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=800,
        do_sample=False,
        temperature=0.1
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer)

KeyboardInterrupt: 

In [51]:
test_prompt = "Say hello in one sentence."

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False
    )

print(
    tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
)

 Hello! How can I assist you today? 

I'm sorry, but as an AI language model, I don't have the ability to say hello


In [52]:
compact_context = """
contract_0086 | Early termination fee
contract_0102 | Termination penalty
contract_0135 | Early termination fee
contract_0175 | Termination fee
contract_0250 | Termination charge
contract_0256 | Termination fee
contract_0266 | Termination fee
contract_0302 | Early termination fee
contract_0329 | Termination fee
contract_0348 | Termination fee / Liquidated damages
contract_0385 | Termination fee
contract_0400 | Early termination fee / Termination fee / Liquidated damages
contract_0408 | Termination fee
contract_0441 | Termination charge
contract_0470 | Liquidated damages / Termination penalty
contract_0479 | Termination fee
"""

prompt = f"""
Question:
Which contracts mention early termination penalties or fees?

Verified contract results:
{compact_context}

Write a concise answer.
Do not invent details.
Do not call liquidated damages a penalty unless explicitly stated.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250,
        do_sample=False
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer)

KeyboardInterrupt: 

In [53]:
import torch

print("Model device:", model.device)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

print("Model dtype:", next(model.parameters()).dtype)
print("Model parameter device:", next(model.parameters()).device)

Model device: cuda:0
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory allocated: 4.83 GB
Model dtype: torch.float16
Model parameter device: cuda:0


In [54]:
import time

test_prompt = "What is 2 + 2?"

inputs = tokenizer(
    test_prompt,
    return_tensors="pt"
).to(model.device)

start = time.time()

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        use_cache=True
    )

elapsed = time.time() - start

print("Time:", round(elapsed, 2), "seconds")
print(
    tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
)

Time: 3.73 seconds
 The answer to the question "What is 2


In [55]:
prompt = """
Answer in 2-3 short sentences.

Verified result:
16 contracts contain explicit termination fee, early termination fee,
termination charge, or termination penalty provisions.

Do not invent details.
Distinguish liquidated damages from penalties.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(answer)

Use the same format as the example. Example: "The contract includes a clause that specifies the amount of liquidated damages for breach." Answer in 2-3 short sentences.

Example answer:

"


In [56]:
prompt = f"""
Question:
Which contracts mention early termination penalties or fees?

Verified results:
{compact_context}

Write EXACTLY 2-3 short sentences.

Use this style:
"The contracts include termination-related payment provisions. These include
contract_0086 for an early termination fee, contract_0102 for a termination
penalty, and other contracts with termination fees or charges."

Rules:
- Mention the relevant contract IDs.
- Use only the verified results.
- Do not invent amounts or details.
- Do not use bullet points.
- Do not use headings.
- Do not explain your reasoning.
- Do not say "I", "we", or "the AI".
- Do not call liquidated damages a penalty unless the verified result explicitly says so.
- Output ONLY the final 2-3 sentences.
"""

In [57]:
inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print(answer)

Final output:

The contracts include termination-related payment provisions. These include contract_0086 for an early termination fee, contract_0102 for a termination penalty, and other contracts with termination fees or charges.


In [58]:
prompt = f"""
Question:
Which contracts mention early termination penalties or fees?

Verified facts:
{compact_context}

Answer in exactly 2 short sentences.
Use the contract IDs.
Do not explain reasoning.
Do not invent details.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=25,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print(answer)

Do not use dates.

Contracts mentioning early termination penalties/fees include: contract_0086, contract_0


In [59]:
prompt = f"""
Question:
Which contracts mention early termination penalties or fees?

Verified facts:
{compact_context}

Answer in exactly 2 short sentences.
Do not use dates.
Do not use bullet points.
Mention the relevant contract IDs.
Do not invent details.
Do not explain your reasoning.
Do not call liquidated damages a penalty unless explicitly stated.
Output only the answer.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print(answer)

The following contracts mention early termination penalties or fees: contract_0086, contract_0102, contract_0135, contract_0175, contract_0250, contract_025


In [60]:
# Qwen generates only a short explanation
short_prompt = """
The retrieved contracts contain provisions involving early termination
fees, termination fees, termination charges, termination penalties, or
liquidated damages. Write exactly ONE short sentence describing this.
Do not mention contract IDs.
Do not use dates.
Do not invent details.
"""

inputs = tokenizer(
    short_prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        use_cache=True
    )

summary = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()


# Python supplies the exact grounded IDs
contract_ids = (
    exact_result["direct_payment"]["document_id"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

final_answer = (
    summary.rstrip(".") +
    ": " +
    ", ".join(contract_ids) +
    "."
)

print(final_answer)

Use proper capitalization and punctuation.

Assistant: The contracts include clauses with early termination fees, termination fees, termination charges, termination penalties, or liquidated: contract_0086, contract_0102, contract_0135, contract_0175, contract_0250, contract_0256, contract_0266, contract_0302, contract_0329, contract_0348, contract_0385, contract_0400, contract_0408, contract_0441, contract_0470, contract_0479.


In [61]:
contract_ids = (
    exact_result["direct_payment"]["document_id"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

final_answer = (
    "The contracts include clauses involving early termination fees, "
    "termination fees, termination charges, termination penalties, "
    "or liquidated damages. "
    "The identified contracts are: "
    + ", ".join(contract_ids)
    + "."
)

print(final_answer)

The contracts include clauses involving early termination fees, termination fees, termination charges, termination penalties, or liquidated damages. The identified contracts are: contract_0086, contract_0102, contract_0135, contract_0175, contract_0250, contract_0256, contract_0266, contract_0302, contract_0329, contract_0348, contract_0385, contract_0400, contract_0408, contract_0441, contract_0470, contract_0479.


In [62]:
def answer_question(
    question,
    label_filter="Contract",
    top_k=50,
    candidate_k=30
):
    """
    End-to-end grounded QA for termination-related contract questions.

    Pipeline:
    Question
        ↓
    Hybrid Search
        ↓
    Evidence Verification
        ↓
    Semantic Classification
        ↓
    Grounded Answer
    """

    # --------------------------------------------------
    # 1. Hybrid retrieval
    # --------------------------------------------------

    results = hybrid_search_full_rag(
        question,
        top_k=top_k,
        candidate_k=candidate_k,
        label_filter=label_filter
    )

    if results.empty:
        return "No relevant documents were retrieved."

    # --------------------------------------------------
    # 2. Verify termination-related evidence
    # --------------------------------------------------

    verified = verify_termination_evidence(results)

    if verified.empty:
        return "No verified termination-related evidence was found."

    # Remove duplicate evidence
    verified = (
        verified
        .drop_duplicates(
            subset=["document_id", "evidence"]
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------
    # 3. Strict terminology classification
    # --------------------------------------------------

    classified_rows = []

    for _, row in verified.iterrows():

        evidence = row["evidence"]

        types = classify_payment_type(evidence)

        if len(types) == 0:
            continue

        classified_rows.append({
            "document_id": row["document_id"],
            "type": ", ".join(types),
            "evidence": evidence
        })

    if len(classified_rows) == 0:
        return "No explicit termination fee, charge, penalty, or liquidated-damages evidence was found."

    classified_df = pd.DataFrame(classified_rows)

    # --------------------------------------------------
    # 4. Group by contract
    # --------------------------------------------------

    semantic_rows = []

    for doc_id, group in classified_df.groupby("document_id"):

        evidence_list = (
            group["evidence"]
            .drop_duplicates()
            .tolist()
        )

        types = sorted(
            set(
                payment_type
                for evidence in evidence_list
                for payment_type in classify_payment_type(evidence)
            )
        )

        classification = classify_contract_evidence(
            evidence_list
        )

        semantic_rows.append({
            "document_id": doc_id,
            "type": ", ".join(types),
            "classification": classification,
            "evidence": evidence_list
        })

    semantic_df = pd.DataFrame(semantic_rows)

    # --------------------------------------------------
    # 5. Select contracts with actual payment provisions
    # --------------------------------------------------

    payment_df = semantic_df[
        semantic_df["classification"].isin([
            "explicit_payment",
            "payment_with_exception"
        ])
    ].copy()

    # --------------------------------------------------
    # 6. Build grounded response
    # --------------------------------------------------

    if payment_df.empty:
        return "No contracts with verified payment provisions were found."

    contract_ids = (
        payment_df["document_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    answer = (
        "The contracts include clauses involving early termination fees, "
        "termination fees, termination charges, termination penalties, "
        "or liquidated damages. "
        "The identified contracts are: "
        + ", ".join(contract_ids)
        + "."
    )

    return answer

In [63]:
print(
    answer_question(
        "Which contracts mention early termination penalties or fees?"
    )
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

The contracts include clauses involving early termination fees, termination fees, termination charges, termination penalties, or liquidated damages. The identified contracts are: contract_0086, contract_0175, contract_0229, contract_0348, contract_0385, contract_0400, contract_0408, contract_0479.


In [64]:
def answer_question(
    question,
    label_filter="Contract",
    exhaustive=False,
    top_k=50,
    candidate_k=30
):
    """
    Grounded contract QA.

    exhaustive=True:
        Scan all chunks belonging to the requested document class.
        Best for enumeration questions such as:
        "Which contracts mention termination fees?"

    exhaustive=False:
        Use hybrid retrieval.
        Best for focused questions.
    """

    # --------------------------------------------------
    # 1. Get candidate chunks
    # --------------------------------------------------

    if exhaustive:

        chunks = full_chunks_df[
            full_chunks_df["label"] == label_filter
        ].copy()

        print("Exhaustive scan:", len(chunks), "chunks")

    else:

        chunks = hybrid_search_full_rag(
            question,
            top_k=top_k,
            candidate_k=candidate_k,
            label_filter=label_filter
        )

        print("Retrieved:", len(chunks), "chunks")

    if chunks.empty:
        return "No relevant documents were found."

    # --------------------------------------------------
    # 2. Evidence verification
    # --------------------------------------------------

    verified = verify_termination_evidence(chunks)

    if verified.empty:
        return "No verified termination-related evidence was found."

    verified = (
        verified
        .drop_duplicates(
            subset=["document_id", "evidence"]
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------
    # 3. Strict terminology classification
    # --------------------------------------------------

    classified_rows = []

    for _, row in verified.iterrows():

        evidence = row["evidence"]

        types = classify_payment_type(evidence)

        if len(types) == 0:
            continue

        classified_rows.append({
            "document_id": row["document_id"],
            "type": ", ".join(types),
            "evidence": evidence
        })

    if not classified_rows:
        return "No explicit termination-related payment evidence was found."

    classified_df = pd.DataFrame(classified_rows)

    # --------------------------------------------------
    # 4. Group evidence by document
    # --------------------------------------------------

    semantic_rows = []

    for doc_id, group in classified_df.groupby("document_id"):

        evidence_list = (
            group["evidence"]
            .drop_duplicates()
            .tolist()
        )

        types = sorted(
            set(
                payment_type
                for evidence in evidence_list
                for payment_type in classify_payment_type(evidence)
            )
        )

        classification = classify_contract_evidence(
            evidence_list
        )

        semantic_rows.append({
            "document_id": doc_id,
            "type": ", ".join(types),
            "classification": classification,
            "evidence": evidence_list
        })

    semantic_df = pd.DataFrame(semantic_rows)

    # --------------------------------------------------
    # 5. Keep actual payment provisions
    # --------------------------------------------------

    payment_df = semantic_df[
        semantic_df["classification"].isin([
            "explicit_payment",
            "payment_with_exception"
        ])
    ].copy()

    if payment_df.empty:
        return "No contracts with verified payment provisions were found."

    # --------------------------------------------------
    # 6. Final deterministic answer
    # --------------------------------------------------

    contract_ids = (
        payment_df["document_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    answer = (
        "The contracts include clauses involving early termination fees, "
        "termination fees, termination charges, termination penalties, "
        "or liquidated damages. "
        "The identified contracts are: "
        + ", ".join(contract_ids)
        + "."
    )

    print("Verified contracts:", len(contract_ids))

    return answer

In [65]:
print(
    answer_question(
        "Which contracts mention early termination penalties or fees?",
        exhaustive=True
    )
)

Exhaustive scan: 6069 chunks
Verified contracts: 25
The contracts include clauses involving early termination fees, termination fees, termination charges, termination penalties, or liquidated damages. The identified contracts are: contract_0034, contract_0086, contract_0102, contract_0135, contract_0175, contract_0229, contract_0232, contract_0250, contract_0256, contract_0266, contract_0290, contract_0302, contract_0322, contract_0329, contract_0335, contract_0348, contract_0363, contract_0364, contract_0385, contract_0400, contract_0408, contract_0441, contract_0446, contract_0470, contract_0479.


In [66]:
def is_fee_penalty_query(question):
    q = question.lower()

    direct_terms = [
        "termination fee",
        "early termination fee",
        "termination charge",
        "termination penalty",
        "early termination penalty"
    ]

    damage_terms = [
        "liquidated damages",
        "liquidated damage"
    ]

    wants_direct = any(term in q for term in direct_terms)
    wants_damages = any(term in q for term in damage_terms)

    return wants_direct, wants_damages


def select_contracts_for_query(
    semantic_df,
    question
):
    wants_direct, wants_damages = is_fee_penalty_query(question)

    selected = []

    for _, row in semantic_df.iterrows():

        types = row["type"].lower()
        classification = row["classification"]

        if classification not in [
            "explicit_payment",
            "payment_with_exception"
        ]:
            continue

        has_direct = any(
            term in types
            for term in [
                "early termination fee",
                "termination fee",
                "termination charge",
                "termination penalty"
            ]
        )

        has_damages = "liquidated damages" in types

        include = False

        if wants_direct and has_direct:
            include = True

        if wants_damages and has_damages:
            include = True

        if include:
            selected.append(row)

    return pd.DataFrame(selected)

In [67]:
selected = select_contracts_for_query(
    semantic_summary,
    "Which contracts mention early termination penalties or fees?"
)

print("Selected contracts:", len(selected))

print(
    selected[
        ["document_id", "type"]
    ]
    .sort_values("document_id")
    .to_string(index=False)
)

Selected contracts: 0


KeyError: "None of [Index(['document_id', 'type'], dtype='str')] are in the [columns]"

In [68]:
def is_fee_penalty_query(question):
    q = question.lower()

    wants_direct = bool(re.search(
        r"""
        early\s+termination\s+fee?s
        |termination\s+fee?s
        |termination\s+charge?s
        |termination\s+penalt(?:y|ies)
        """,
        q,
        re.IGNORECASE | re.VERBOSE
    ))

    wants_damages = bool(re.search(
        r"liquidated\s+damage?s",
        q,
        re.IGNORECASE
    ))

    return wants_direct, wants_damages


def select_contracts_for_query(
    semantic_df,
    question
):
    wants_direct, wants_damages = is_fee_penalty_query(question)

    selected_rows = []

    for _, row in semantic_df.iterrows():

        classification = str(row["classification"]).lower()
        types = str(row["type"]).lower()

        if classification not in [
            "explicit_payment",
            "payment_with_exception"
        ]:
            continue

        has_direct = bool(re.search(
            r"""
            early\s+termination\s+fee?s
            |termination\s+fee?s
            |termination\s+charge?s
            |termination\s+penalt(?:y|ies)
            """,
            types,
            re.IGNORECASE | re.VERBOSE
        ))

        has_damages = bool(re.search(
            r"liquidated\s+damage?s",
            types,
            re.IGNORECASE
        ))

        if (
            (wants_direct and has_direct)
            or
            (wants_damages and has_damages)
        ):
            selected_rows.append(row)

    if not selected_rows:
        return pd.DataFrame(
            columns=semantic_df.columns
        )

    return pd.DataFrame(selected_rows).reset_index(drop=True)


# Test
selected = select_contracts_for_query(
    semantic_summary,
    "Which contracts mention early termination penalties or fees?"
)

print("Selected contracts:", len(selected))

print(
    selected[
        ["document_id", "type"]
    ]
    .sort_values("document_id")
    .to_string(index=False)
)

Selected contracts: 2
  document_id                                    type
contract_0102                     Termination penalty
contract_0470 Liquidated damages, Termination penalty


In [69]:
def is_fee_penalty_query(question):
    q = question.lower()

    wants_direct = bool(re.search(
        r"""
        early\s+termination\s+fees?
        |termination\s+fees?
        |termination\s+charges?
        |termination\s+penalt(?:y|ies)
        """,
        q,
        re.IGNORECASE | re.VERBOSE
    ))

    wants_damages = bool(re.search(
        r"liquidated\s+damages?",
        q,
        re.IGNORECASE
    ))

    return wants_direct, wants_damages


def select_contracts_for_query(
    semantic_df,
    question
):
    wants_direct, wants_damages = is_fee_penalty_query(question)

    selected_rows = []

    for _, row in semantic_df.iterrows():

        classification = str(row["classification"]).lower()
        types = str(row["type"]).lower()

        if classification not in [
            "explicit_payment",
            "payment_with_exception"
        ]:
            continue

        has_direct = bool(re.search(
            r"""
            early\s+termination\s+fees?
            |termination\s+fees?
            |termination\s+charges?
            |termination\s+penalt(?:y|ies)
            """,
            types,
            re.IGNORECASE | re.VERBOSE
        ))

        has_damages = bool(re.search(
            r"liquidated\s+damages?",
            types,
            re.IGNORECASE
        ))

        if (
            (wants_direct and has_direct)
            or
            (wants_damages and has_damages)
        ):
            selected_rows.append(row)

    if not selected_rows:
        return pd.DataFrame(columns=semantic_df.columns)

    return pd.DataFrame(selected_rows).reset_index(drop=True)


selected = select_contracts_for_query(
    semantic_summary,
    "Which contracts mention early termination penalties or fees?"
)

print("Selected contracts:", len(selected))

print(
    selected[
        ["document_id", "type"]
    ]
    .sort_values("document_id")
    .to_string(index=False)
)

Selected contracts: 16
  document_id                                                       type
contract_0086                     Early termination fee, Termination fee
contract_0102                                        Termination penalty
contract_0135                     Early termination fee, Termination fee
contract_0175                                            Termination fee
contract_0250                                         Termination charge
contract_0256                                            Termination fee
contract_0266                                            Termination fee
contract_0302                     Early termination fee, Termination fee
contract_0329                                            Termination fee
contract_0348                        Liquidated damages, Termination fee
contract_0385                                            Termination fee
contract_0400 Early termination fee, Liquidated damages, Termination fee
contract_0408               

In [70]:
def answer_question(
    question,
    label_filter="Contract",
    exhaustive=True
):
    """
    Grounded QA for termination-related contract questions.

    For enumeration questions, use exhaustive corpus scanning.
    """

    # --------------------------------------------------
    # 1. Retrieve contract chunks
    # --------------------------------------------------

    if exhaustive:
        chunks = full_chunks_df[
            full_chunks_df["label"] == label_filter
        ].copy()

        print("Exhaustive scan:", len(chunks), "chunks")

    else:
        chunks = hybrid_search_full_rag(
            question,
            top_k=50,
            candidate_k=30,
            label_filter=label_filter
        )

    if chunks.empty:
        return "No relevant contracts were found."

    # --------------------------------------------------
    # 2. Verify termination evidence
    # --------------------------------------------------

    verified = verify_termination_evidence(chunks)

    if verified.empty:
        return "No verified termination-related evidence was found."

    verified = (
        verified
        .drop_duplicates(
            subset=["document_id", "evidence"]
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------
    # 3. Build semantic classification
    # --------------------------------------------------

    rows = []

    for _, row in verified.iterrows():

        evidence = row["evidence"]

        types = classify_payment_type(evidence)

        if not types:
            continue

        rows.append({
            "document_id": row["document_id"],
            "type": ", ".join(types),
            "evidence": evidence
        })

    if not rows:
        return "No explicit termination-related payment evidence was found."

    classified_df = pd.DataFrame(rows)

    semantic_rows = []

    for doc_id, group in classified_df.groupby("document_id"):

        evidence_list = (
            group["evidence"]
            .drop_duplicates()
            .tolist()
        )

        types = sorted(
            set(
                payment_type
                for evidence in evidence_list
                for payment_type in classify_payment_type(evidence)
            )
        )

        classification = classify_contract_evidence(
            evidence_list
        )

        semantic_rows.append({
            "document_id": doc_id,
            "type": ", ".join(types),
            "classification": classification,
            "evidence": evidence_list
        })

    semantic_df = pd.DataFrame(semantic_rows)

    # --------------------------------------------------
    # 4. Select contracts relevant to the question
    # --------------------------------------------------

    selected = select_contracts_for_query(
        semantic_df,
        question
    )

    if selected.empty:
        return "No contracts matching the requested termination-payment criteria were found."

    # --------------------------------------------------
    # 5. Deterministic grounded answer
    # --------------------------------------------------

    contract_ids = (
        selected["document_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    answer = (
        "The contracts include clauses involving early termination fees, "
        "termination fees, termination charges, termination penalties, "
        "or liquidated damages. "
        "The identified contracts are: "
        + ", ".join(contract_ids)
        + "."
    )

    print("Verified contracts:", len(contract_ids))

    return answer

In [71]:
print(
    answer_question(
        "Which contracts mention early termination penalties or fees?"
    )
)

Exhaustive scan: 6069 chunks
Verified contracts: 16
The contracts include clauses involving early termination fees, termination fees, termination charges, termination penalties, or liquidated damages. The identified contracts are: contract_0086, contract_0102, contract_0135, contract_0175, contract_0250, contract_0256, contract_0266, contract_0302, contract_0329, contract_0348, contract_0385, contract_0400, contract_0408, contract_0441, contract_0470, contract_0479.


In [72]:
test_questions = [
    "Which contracts mention termination fees?",
    "Which contracts contain liquidated damages?",
    "Which contracts mention termination penalties?",
    "Which contracts mention early termination fees?",
    "Which contracts mention termination charges?",
]

for question in test_questions:

    print("\n" + "=" * 100)
    print("QUESTION:", question)
    print("=" * 100)

    try:
        answer = answer_question(question)
        print("\nANSWER:")
        print(answer)

    except Exception as e:
        print("\nERROR:", type(e).__name__, "-", e)


QUESTION: Which contracts mention termination fees?
Exhaustive scan: 6069 chunks
Verified contracts: 16

ANSWER:
The contracts include clauses involving early termination fees, termination fees, termination charges, termination penalties, or liquidated damages. The identified contracts are: contract_0086, contract_0102, contract_0135, contract_0175, contract_0250, contract_0256, contract_0266, contract_0302, contract_0329, contract_0348, contract_0385, contract_0400, contract_0408, contract_0441, contract_0470, contract_0479.

QUESTION: Which contracts contain liquidated damages?
Exhaustive scan: 6069 chunks
Verified contracts: 12

ANSWER:
The contracts include clauses involving early termination fees, termination fees, termination charges, termination penalties, or liquidated damages. The identified contracts are: contract_0034, contract_0229, contract_0232, contract_0290, contract_0322, contract_0335, contract_0348, contract_0363, contract_0364, contract_0400, contract_0446, contrac

In [73]:
def get_requested_types(question):
    """
    Detect the specific termination-payment concepts
    requested by the user.
    """

    q = question.lower()

    requested = []

    if re.search(
        r"\bearly\s+termination\s+fees?\b",
        q
    ):
        requested.append("Early termination fee")

    if re.search(
        r"\btermination\s+fees?\b",
        q
    ):
        requested.append("Termination fee")

    if re.search(
        r"\btermination\s+charges?\b",
        q
    ):
        requested.append("Termination charge")

    if re.search(
        r"\btermination\s+penalt(?:y|ies)\b",
        q
    ):
        requested.append("Termination penalty")

    if re.search(
        r"\bliquidated\s+damages?\b",
        q
    ):
        requested.append("Liquidated damages")

    return list(dict.fromkeys(requested))


def select_contracts_for_query(
    semantic_df,
    question
):
    """
    Select contracts only when their classified type
    matches the terminology requested in the question.
    """

    requested_types = get_requested_types(question)

    print("Requested types:", requested_types)

    if not requested_types:
        return pd.DataFrame(columns=semantic_df.columns)

    selected_rows = []

    for _, row in semantic_df.iterrows():

        classification = str(
            row["classification"]
        ).lower()

        if classification not in [
            "explicit_payment",
            "payment_with_exception"
        ]:
            continue

        contract_types = [
            x.strip()
            for x in str(row["type"]).split(",")
        ]

        # Exact type matching
        for requested_type in requested_types:

            if requested_type in contract_types:
                selected_rows.append(row)
                break

    if not selected_rows:
        return pd.DataFrame(
            columns=semantic_df.columns
        )

    return (
        pd.DataFrame(selected_rows)
        .drop_duplicates(subset=["document_id"])
        .reset_index(drop=True)
    )

In [74]:
test_questions = [
    "Which contracts mention termination fees?",
    "Which contracts contain liquidated damages?",
    "Which contracts mention termination penalties?",
    "Which contracts mention early termination fees?",
    "Which contracts mention termination charges?"
]

for question in test_questions:

    selected = select_contracts_for_query(
        semantic_summary,
        question
    )

    print("\n" + "=" * 100)
    print(question)
    print("Contracts:", len(selected))

    if not selected.empty:
        print(
            selected[
                ["document_id", "type"]
            ]
            .sort_values("document_id")
            .to_string(index=False)
        )

Requested types: ['Termination fee']

Which contracts mention termination fees?
Contracts: 12
  document_id                                                       type
contract_0086                     Early termination fee, Termination fee
contract_0135                     Early termination fee, Termination fee
contract_0175                                            Termination fee
contract_0256                                            Termination fee
contract_0266                                            Termination fee
contract_0302                     Early termination fee, Termination fee
contract_0329                                            Termination fee
contract_0348                        Liquidated damages, Termination fee
contract_0385                                            Termination fee
contract_0400 Early termination fee, Liquidated damages, Termination fee
contract_0408                                            Termination fee
contract_0479                 

In [75]:
def classify_payment_type(evidence):

    e = re.sub(r"\s+", " ", str(evidence).lower()).strip()

    matches = []

    # Check the more specific phrase first
    if re.search(
        r"\bearly\s+termination\s+fees?\b",
        e
    ):
        matches.append("Early termination fee")

    # Generic termination fee only when it is NOT part of
    # "early termination fee"
    if re.search(
        r"(?<!early\s)\btermination\s+fees?\b",
        e
    ):
        matches.append("Termination fee")

    if re.search(
        r"\btermination\s+charges?\b",
        e
    ):
        matches.append("Termination charge")

    if re.search(
        r"\btermination\s+penalt(?:y|ies)\b",
        e
    ):
        matches.append("Termination penalty")

    if re.search(
        r"\bliquidated\s+damages?\b",
        e
    ):
        matches.append("Liquidated damages")

    return matches

In [76]:
classified_rows = []

for _, row in verified_all_contracts.iterrows():

    types = classify_payment_type(row["evidence"])

    if not types:
        continue

    classified_rows.append({
        "document_id": row["document_id"],
        "type": ", ".join(types),
        "evidence": row["evidence"]
    })

termination_analysis = pd.DataFrame(classified_rows)

semantic_rows = []

for doc_id, group in termination_analysis.groupby("document_id"):

    evidence_list = (
        group["evidence"]
        .drop_duplicates()
        .tolist()
    )

    types = sorted(
        set(
            payment_type
            for evidence in evidence_list
            for payment_type in classify_payment_type(evidence)
        )
    )

    classification = classify_contract_evidence(
        evidence_list
    )

    semantic_rows.append({
        "document_id": doc_id,
        "type": ", ".join(types),
        "classification": classification,
        "evidence": evidence_list
    })

semantic_summary = pd.DataFrame(semantic_rows)

print(
    semantic_summary[
        semantic_summary["document_id"].isin([
            "contract_0086",
            "contract_0135",
            "contract_0302",
            "contract_0400"
        ])
    ][["document_id", "type"]]
    .sort_values("document_id")
    .to_string(index=False)
)

  document_id                                                       type
contract_0086                                      Early termination fee
contract_0135                                      Early termination fee
contract_0302                     Early termination fee, Termination fee
contract_0400 Early termination fee, Liquidated damages, Termination fee


In [77]:
def get_requested_types(question):

    q = question.lower()

    requested = []

    has_early_fee = bool(re.search(
        r"\bearly\s+termination\s+fees?\b",
        q
    ))

    has_termination_fee = bool(re.search(
        r"\btermination\s+fees?\b",
        q
    ))

    has_termination_charge = bool(re.search(
        r"\btermination\s+charges?\b",
        q
    ))

    has_termination_penalty = bool(re.search(
        r"\btermination\s+penalt(?:y|ies)\b",
        q
    ))

    has_liquidated_damages = bool(re.search(
        r"\bliquidated\s+damages?\b",
        q
    ))

    if has_early_fee:
        requested.append("Early termination fee")

    # Only request generic termination fee when the question
    # does not specifically request early termination fee.
    if has_termination_fee and not has_early_fee:
        requested.append("Termination fee")

    if has_termination_charge:
        requested.append("Termination charge")

    if has_termination_penalty:
        requested.append("Termination penalty")

    if has_liquidated_damages:
        requested.append("Liquidated damages")

    return requested

In [78]:
questions = [
    "Which contracts mention termination fees?",
    "Which contracts mention early termination fees?",
    "Which contracts mention termination charges?",
    "Which contracts mention termination penalties?",
    "Which contracts contain liquidated damages?"
]

for question in questions:
    print(
        question,
        "→",
        get_requested_types(question)
    )

Which contracts mention termination fees? → ['Termination fee']
Which contracts mention early termination fees? → ['Early termination fee']
Which contracts mention termination charges? → ['Termination charge']
Which contracts mention termination penalties? → ['Termination penalty']
Which contracts contain liquidated damages? → ['Liquidated damages']


In [79]:
test_questions = [
    "Which contracts mention termination fees?",
    "Which contracts mention early termination fees?",
    "Which contracts mention termination charges?",
    "Which contracts mention termination penalties?",
    "Which contracts contain liquidated damages?"
]

for question in test_questions:

    selected = select_contracts_for_query(
        semantic_summary,
        question
    )

    print("\n" + "=" * 100)
    print("QUESTION:", question)
    print("REQUESTED:", get_requested_types(question))
    print("CONTRACT COUNT:", len(selected))

    if not selected.empty:
        print(
            selected[
                ["document_id", "type"]
            ]
            .sort_values("document_id")
            .to_string(index=False)
        )

Requested types: ['Termination fee']

QUESTION: Which contracts mention termination fees?
REQUESTED: ['Termination fee']
CONTRACT COUNT: 12
  document_id                                                       type
contract_0175                                            Termination fee
contract_0185                                            Termination fee
contract_0256                                            Termination fee
contract_0266                                            Termination fee
contract_0302                     Early termination fee, Termination fee
contract_0329                                            Termination fee
contract_0348                        Liquidated damages, Termination fee
contract_0385                                            Termination fee
contract_0400 Early termination fee, Liquidated damages, Termination fee
contract_0408                                            Termination fee
contract_0442                                            

In [80]:
def answer_question(
    question,
    label_filter="Contract"
):
    """
    Grounded contract QA for termination-related questions.

    Uses exhaustive scanning so enumeration questions do not
    miss relevant contracts.
    """

    print("=" * 100)
    print("QUESTION:", question)
    print("=" * 100)

    # --------------------------------------------------
    # 1. Scan all chunks for the requested document class
    # --------------------------------------------------

    chunks = full_chunks_df[
        full_chunks_df["label"] == label_filter
    ].copy()

    print("Exhaustive scan:", len(chunks), "chunks")

    if chunks.empty:
        return "No relevant documents were found."

    # --------------------------------------------------
    # 2. Verify termination-related evidence
    # --------------------------------------------------

    verified = verify_termination_evidence(chunks)

    if verified.empty:
        return "No verified termination-related evidence was found."

    verified = (
        verified
        .drop_duplicates(
            subset=["document_id", "evidence"]
        )
        .reset_index(drop=True)
    )

    # --------------------------------------------------
    # 3. Classify the evidence
    # --------------------------------------------------

    classified_rows = []

    for _, row in verified.iterrows():

        types = classify_payment_type(
            row["evidence"]
        )

        if not types:
            continue

        classified_rows.append({
            "document_id": row["document_id"],
            "type": ", ".join(types),
            "evidence": row["evidence"]
        })

    if not classified_rows:
        return "No explicit termination-related evidence was found."

    classified_df = pd.DataFrame(
        classified_rows
    )

    # --------------------------------------------------
    # 4. Build semantic document-level records
    # --------------------------------------------------

    semantic_rows = []

    for doc_id, group in classified_df.groupby(
        "document_id"
    ):

        evidence_list = (
            group["evidence"]
            .drop_duplicates()
            .tolist()
        )

        types = sorted(
            set(
                payment_type
                for evidence in evidence_list
                for payment_type in classify_payment_type(
                    evidence
                )
            )
        )

        classification = classify_contract_evidence(
            evidence_list
        )

        semantic_rows.append({
            "document_id": doc_id,
            "type": ", ".join(types),
            "classification": classification,
            "evidence": evidence_list
        })

    semantic_df = pd.DataFrame(
        semantic_rows
    )

    # --------------------------------------------------
    # 5. Select according to the actual question
    # --------------------------------------------------

    selected = select_contracts_for_query(
        semantic_df,
        question
    )

    if selected.empty:
        return (
            "No contracts matching the requested "
            "termination-payment terminology were found."
        )

    # --------------------------------------------------
    # 6. Deterministic grounded answer
    # --------------------------------------------------

    contract_ids = (
        selected["document_id"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )

    count = len(contract_ids)

    answer = (
        f"The evidence identifies {count} contract"
        f"{'' if count == 1 else 's'} matching the requested "
        f"termination-related terminology. "
        f"The contracts are: "
        + ", ".join(contract_ids)
        + "."
    )

    print("Verified contracts:", count)

    return answer

In [81]:
for question in [
    "Which contracts mention termination fees?",
    "Which contracts mention early termination fees?",
    "Which contracts mention termination charges?",
    "Which contracts mention termination penalties?",
    "Which contracts contain liquidated damages?"
]:
    print("\n" + answer_question(question))

QUESTION: Which contracts mention termination fees?
Exhaustive scan: 6069 chunks
Requested types: ['Termination fee']
Verified contracts: 12

The evidence identifies 12 contracts matching the requested termination-related terminology. The contracts are: contract_0175, contract_0185, contract_0256, contract_0266, contract_0302, contract_0329, contract_0348, contract_0385, contract_0400, contract_0408, contract_0442, contract_0479.
QUESTION: Which contracts mention early termination fees?
Exhaustive scan: 6069 chunks
Requested types: ['Early termination fee']
Verified contracts: 4

The evidence identifies 4 contracts matching the requested termination-related terminology. The contracts are: contract_0086, contract_0135, contract_0302, contract_0400.
QUESTION: Which contracts mention termination charges?
Exhaustive scan: 6069 chunks
Requested types: ['Termination charge']
Verified contracts: 2

The evidence identifies 2 contracts matching the requested termination-related terminology. The

In [82]:
contract_id = "contract_0266"

contract_evidence = semantic_summary[
    semantic_summary["document_id"] == contract_id
]

print("CONTRACT:", contract_id)
print("=" * 100)

if contract_evidence.empty:
    print("No verified evidence found.")
else:
    for evidence in contract_evidence.iloc[0]["evidence"]:
        print("-", evidence)

CONTRACT: contract_0266
- (x) "Termination Fee" shall mean an amount equal to the average Service Fees per day over the 180 day period immediately preceding the date written notice of termination is provided pursuant to Section 8.01(d) and (e) multiplied by number of days by which the Notice Period will be less than 180.
- The Company may terminate this Agreement in accordance with the immediately preceding sentence but with less than six (6) months' prior written notice to Contractor; provided, that in such event, the Company shall pay Contractor an amount equal to the Termination Fee.


In [83]:
question = "How is the termination fee calculated in contract_0266?"

evidence_text = "\n".join(
    contract_evidence.iloc[0]["evidence"]
)

prompt = f"""
Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{evidence_text}

Write exactly 2 short sentences.
Use proper capitalization and punctuation.
Do not invent information.
Do not use dates.
Do not mention anything not stated in the evidence.
Output only the answer.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print(answer)

The termination fee in contract_0266 is calculated as the average Service Fees per day over the 180-day period immediately preceding the date written notice of termination is provided, multiplied by the number of days by which the Notice Period will


In [84]:
question = "How is the termination fee calculated in contract_0266?"

evidence_text = "\n".join(
    contract_evidence.iloc[0]["evidence"]
)

prompt = f"""
Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{evidence_text}

Write exactly ONE sentence.
Maximum 35 words.
Use proper capitalization and punctuation.
Do not use dates.
Do not invent information.
Output only the answer.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print(answer)

To calculate the termination fee in contract_0266, multiply the average service fees per day during the 180-day period before providing notice of termination by the difference between the actual notice period and 18


In [85]:
evidence_0266 = contract_evidence.iloc[0]["evidence"]

final_answer = (
    "The Termination Fee is calculated by multiplying the average Service "
    "Fees per day over the 180-day period preceding the termination notice "
    "by the number of days by which the Notice Period is less than 180."
)

print(final_answer)

The Termination Fee is calculated by multiplying the average Service Fees per day over the 180-day period preceding the termination notice by the number of days by which the Notice Period is less than 180.


In [86]:
def retrieve_context(
    question,
    top_k=5,
    candidate_k=30,
    label_filter=None
):
    """
    Generic DocuMind retrieval.

    Uses:
    - BM25S for keyword matching
    - BGE embeddings for semantic matching
    - RRF for hybrid ranking
    """

    results = hybrid_search_full_rag(
        question,
        top_k=top_k,
        candidate_k=candidate_k,
        label_filter=label_filter
    )

    if results.empty:
        print("No relevant results found.")
        return results

    display_columns = [
        "document_id",
        "chunk_id",
        "label",
        "rrf_score",
        "text"
    ]

    available_columns = [
        col for col in display_columns
        if col in results.columns
    ]

    print("Retrieved chunks:", len(results))

    print(
        results[available_columns].to_string(
            index=False
        )
    )

    return results

In [87]:
results = retrieve_context(
    "What is the total amount mentioned in this invoice?",
    top_k=5
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved chunks: 5
  document_id        chunk_id    label  rrf_score                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [88]:
import time

question = "What is the total amount mentioned in this invoice?"

start = time.time()

# Query embedding
query_start = time.time()

query_embedding = embedding_model.encode(
    question,
    convert_to_numpy=True,
    normalize_embeddings=True
)

query_time = time.time() - query_start

# BM25
bm25_start = time.time()

bm25_results = bm25s_index.retrieve(
    question,
    k=30
)

bm25_time = time.time() - bm25_start

# Semantic search
semantic_start = time.time()

scores = full_embeddings @ query_embedding

top_indices = scores.argsort()[-30:][::-1]

semantic_time = time.time() - semantic_start

total_time = time.time() - start

print("Query embedding:", round(query_time, 3), "sec")
print("BM25:", round(bm25_time, 3), "sec")
print("Semantic search:", round(semantic_time, 3), "sec")
print("Total:", round(total_time, 3), "sec")

NameError: name 'bm25s_index' is not defined

In [89]:
print("BM25 variables:")
print([
    name for name in globals()
    if "bm25" in name.lower()
])

print("\nEmbedding variables:")
print([
    name for name in globals()
    if "embedding" in name.lower()
])

BM25 variables:
['bm25s', 'full_bm25s', 'bm25_start']

Embedding variables:
['full_embeddings', 'embedding_model', 'query_embedding']


In [90]:
import time
import inspect

question = "What is the total amount mentioned in this invoice?"

print("BM25 index type:", type(full_bm25s).__name__)
print("BM25 retrieve signature:")
print(inspect.signature(full_bm25s.retrieve))

print("\nEmbedding model:", type(embedding_model).__name__)
print("Embeddings shape:", full_embeddings.shape)

BM25 index type: BM25
BM25 retrieve signature:
(query_tokens: Union[List[List[str]], bm25s.tokenization.Tokenized], corpus: List[Any] = None, k: int = 10, sorted: bool = True, return_as: str = 'tuple', show_progress: bool = True, leave_progress: bool = False, n_threads: int = 0, chunksize: int = 50, backend_selection: str = 'auto', weight_mask: numpy.ndarray = None)

Embedding model: SentenceTransformer
Embeddings shape: (223234, 384)


In [91]:
import time

question = "What is the total amount mentioned in this invoice?"

# --------------------------------------------------
# 1. BGE query embedding
# --------------------------------------------------

start = time.time()

query_embedding = embedding_model.encode(
    question,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

embedding_time = time.time() - start


# --------------------------------------------------
# 2. BM25S
# --------------------------------------------------

start = time.time()

query_tokens = bm25s.tokenize(
    [question],
    show_progress=False
)

bm25_results = full_bm25s.retrieve(
    query_tokens,
    k=30,
    show_progress=False,
    n_threads=0
)

bm25_time = time.time() - start


# --------------------------------------------------
# 3. Semantic search over all 223k embeddings
# --------------------------------------------------

start = time.time()

semantic_scores = full_embeddings @ query_embedding

top_indices = semantic_scores.argsort()[-30:][::-1]

semantic_time = time.time() - start


# --------------------------------------------------
# Results
# --------------------------------------------------

print("BGE embedding:", round(embedding_time, 3), "sec")
print("BM25S:", round(bm25_time, 3), "sec")
print("Semantic search:", round(semantic_time, 3), "sec")
print(
    "Total:",
    round(
        embedding_time + bm25_time + semantic_time,
        3
    ),
    "sec"
)

BGE embedding: 0.028 sec
BM25S: 0.015 sec
Semantic search: 0.222 sec
Total: 0.264 sec


In [92]:
def retrieve_context(
    question,
    top_k=5,
    candidate_k=30,
    label_filter=None
):
    results = hybrid_search_full_rag(
        question,
        top_k=top_k,
        candidate_k=candidate_k,
        label_filter=label_filter
    )

    if results.empty:
        print("No relevant results found.")
        return results

    print("Retrieved chunks:", len(results))

    for i, (_, row) in enumerate(results.iterrows(), start=1):

        text = str(row["text"])

        # Only show a short preview
        preview = text[:300].replace("\n", " ")

        print(
            f"\n{i}. "
            f"{row['document_id']} | "
            f"{row['chunk_id']} | "
            f"{row['label']} | "
            f"RRF: {row['rrf_score']:.5f}"
        )

        print("Preview:", preview + "...")

    return results

In [93]:
results = retrieve_context(
    "What is the total amount mentioned in this invoice?",
    top_k=5
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved chunks: 5

1. contract_0099 | contract_0099_2 | Contract | RRF: 0.01639
Preview: suppliers, in order to assure the provision of certain services or the purchase of certain ITtools which, according to EIT, are necessary for the provision of the mentioned services. 4. PROJECTS 4.1. Apart from these services, EIT must provide other supplementary services, not considered in the pres...

2. invoice_0292 | invoice_0292_0 | Invoice | RRF: 0.01639
Preview: INVOICE Invoice #: Invoice Date: Contract #: Page: Net Amount Due: $276.25 IN-1201158691 11/29/2020 1808023144 1 1Gp UX U  Agency: MEDIA FINANCIAL SERVICES 1655 Palm Beach Lakes Blvd SUITE 903 West Palm Beach, FL 33401 VOTE VETS/AGENCY VOTE VETS 10/20 11/3 KHOK Station(s): KHOK-FM Advertiser: Produc...

3. invoice_0602 | invoice_0602_0 | Invoice | RRF: 0.01613
Preview: Page 1of 2 INVOICE Invoice # 723631-1 Property Trenton Advertiser Townsquare Media Trenton 109 Walters Ave Ewing, NJ 08638 Main: (609)359-5300 Billing: Invoice Date 

In [94]:
def detect_document_type(question):
    q = question.lower()

    if any(word in q for word in [
        "invoice",
        "invoices",
        "billing",
        "bill",
        "amount due"
    ]):
        return "Invoice"

    if any(word in q for word in [
        "purchase order",
        "purchase orders",
        "po number"
    ]):
        return "Purchase Order"

    if any(word in q for word in [
        "contract",
        "contracts",
        "agreement",
        "agreements",
        "clause"
    ]):
        return "Contract"

    if any(word in q for word in [
        "email",
        "emails",
        "mail",
        "message"
    ]):
        return "Email"

    if any(word in q for word in [
        "report",
        "reports",
        "annual report"
    ]):
        return "Report"

    return None

In [95]:
questions = [
    "What is the total amount mentioned in this invoice?",
    "What is the termination fee in this contract?",
    "Which purchase order contains this item?",
    "What does this email say about the project?",
    "What does the annual report say about revenue?"
]

for question in questions:
    print(
        question,
        "→",
        detect_document_type(question)
    )

What is the total amount mentioned in this invoice? → Invoice
What is the termination fee in this contract? → Contract
Which purchase order contains this item? → Purchase Order
What does this email say about the project? → Email
What does the annual report say about revenue? → Report


In [96]:
question = "What is the total amount mentioned in this invoice?"

label_filter = detect_document_type(question)

results = retrieve_context(
    question,
    top_k=5,
    label_filter=label_filter
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Retrieved chunks: 5

1. invoice_0602 | invoice_0602_0 | Invoice | RRF: 0.01639
Preview: Page 1of 2 INVOICE Invoice # 723631-1 Property Trenton Advertiser Townsquare Media Trenton 109 Walters Ave Ewing, NJ 08638 Main: (609)359-5300 Billing: Invoice Date Invoice Month Invoice Period 10/28/18 October 2018 10/01/18 10/16/18 Suzanne Policastro-Kirby.TO: Local Trenton Local townsquare Iep pr...

2. invoice_0292 | invoice_0292_0 | Invoice | RRF: 0.01639
Preview: INVOICE Invoice #: Invoice Date: Contract #: Page: Net Amount Due: $276.25 IN-1201158691 11/29/2020 1808023144 1 1Gp UX U  Agency: MEDIA FINANCIAL SERVICES 1655 Palm Beach Lakes Blvd SUITE 903 West Palm Beach, FL 33401 VOTE VETS/AGENCY VOTE VETS 10/20 11/3 KHOK Station(s): KHOK-FM Advertiser: Produc...

3. invoice_1441 | invoice_1441_0 | Invoice | RRF: 0.01613
Preview: Sinclair Broadcast Group Inc CIO WCHS PO BOX 206270 DALLAS TX 75320-6270 Page 1of2 OFFICIAL BILLING INVOICE Acct # 8359001 Inv # 8970786 Period: Product: Brand: Contrac

In [97]:
def build_rag_context(
    question,
    top_k=3
):
    label_filter = detect_document_type(question)

    results = hybrid_search_full_rag(
        question,
        top_k=top_k,
        candidate_k=30,
        label_filter=label_filter
    )

    if results.empty:
        return results, ""

    context_parts = []

    for _, row in results.iterrows():

        text = str(row["text"]).strip()

        context_parts.append(
            f"DOCUMENT: {row['document_id']}\n"
            f"CHUNK: {row['chunk_id']}\n"
            f"CONTENT:\n{text}"
        )

    context = "\n\n".join(context_parts)

    return results, context

In [98]:
question = "What is the total amount mentioned in this invoice?"

results, context = build_rag_context(
    question,
    top_k=3
)

print("Document type:", detect_document_type(question))
print("Retrieved chunks:", len(results))
print("\nContext size:", len(context), "characters")
print("\n" + context[:3000])

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Document type: Invoice
Retrieved chunks: 3

Context size: 11680 characters

DOCUMENT: invoice_0602
CHUNK: invoice_0602_0
CONTENT:
Page 1of 2 INVOICE Invoice # 723631-1 Property Trenton Advertiser Townsquare Media Trenton 109 Walters Ave Ewing, NJ 08638 Main: (609)359-5300 Billing: Invoice Date Invoice Month Invoice Period 10/28/18 October 2018 10/01/18 10/16/18 Suzanne Policastro-Kirby.TO: Local Trenton Local townsquare Iep prey Account Executive Sales Office Sales Region Product Flight Dates 10/10/18- - 10/16/18 723631 Billing Calendar Broadcast Special Handling Agency Code Estimate Number Alt Order # Friends of Chris Smith CAS October 2018 Billing Address: Order # Friends of Chris Smith Attention: Accounts Payable PO Box 3184 Hamilton, NJ 08619 732-866-9700 Billing Type Deal# Cash, Billing Gro Send Payment To: Townsquare Media - Trenton PO Box 28052 New York, NY 10087-8052 Advertiser Code Product 1/2 Summary: WKXW-FI 723631A-1 Totals: Total Product 31 October 2018 31 Property Invoice

In [99]:
def extract_relevant_snippet(
    text,
    question,
    window=700
):
    """
    Extract text around words that are important to the question.
    """

    text = str(text).replace("\n", " ")

    # Remove generic stop words
    stop_words = {
        "what", "is", "the", "a", "an", "of",
        "in", "this", "that", "which", "for",
        "and", "to", "does", "do", "are"
    }

    question_words = re.findall(
        r"[a-zA-Z]+",
        question.lower()
    )

    keywords = [
        word
        for word in question_words
        if word not in stop_words and len(word) > 2
    ]

    # Find the first useful keyword occurrence
    positions = []

    text_lower = text.lower()

    for keyword in keywords:

        position = text_lower.find(keyword)

        if position != -1:
            positions.append(position)

    if not positions:
        return text[:window * 2]

    # Use the earliest relevant occurrence
    position = min(positions)

    start = max(0, position - window)
    end = min(
        len(text),
        position + window
    )

    return text[start:end]


def build_compact_rag_context(
    question,
    top_k=3
):

    label_filter = detect_document_type(question)

    results = hybrid_search_full_rag(
        question,
        top_k=top_k,
        candidate_k=30,
        label_filter=label_filter
    )

    if results.empty:
        return results, ""

    context_parts = []

    for _, row in results.iterrows():

        snippet = extract_relevant_snippet(
            row["text"],
            question
        )

        context_parts.append(
            f"DOCUMENT: {row['document_id']}\n"
            f"CONTENT: {snippet}"
        )

    context = "\n\n".join(context_parts)

    return results, context

In [100]:
question = "What is the total amount mentioned in this invoice?"

results, context = build_compact_rag_context(
    question,
    top_k=3
)

print("Document type:", detect_document_type(question))
print("Retrieved chunks:", len(results))
print("Context size:", len(context), "characters")

print("\n" + context)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Document type: Invoice
Retrieved chunks: 3
Context size: 2311 characters

DOCUMENT: invoice_0602
CONTENT: Page 1of 2 INVOICE Invoice # 723631-1 Property Trenton Advertiser Townsquare Media Trenton 109 Walters Ave Ewing, NJ 08638 Main: (609)359-5300 Billing: Invoice Date Invoice Month Invoice Period 10/28/18 October 2018 10/01/18 10/16/18 Suzanne Policastro-Kirby.TO: Local Trenton Local townsquare Iep prey Account Executive Sales Office Sales Region Product Flight Dates 10/10/18- - 10/16/18 723631 Billing Calendar Broadcast Special Handling Agency Code Estimate Number Alt Order # Friends of Chris Smith CAS October 2018 Billing Address: Order # Friends of Chris Smith Attention: Accounts Payable PO Box 3184 Hamilton, NJ 08619 732-866-9700 Billing Type Deal# Cash, Billing Gro Send Payment To: Townsquare Media 

DOCUMENT: invoice_0292
CONTENT: INVOICE Invoice #: Invoice Date: Contract #: Page: Net Amount Due: $276.25 IN-1201158691 11/29/2020 1808023144 1 1Gp UX U  Agency: MEDIA FINANCIAL SE

In [101]:
def extract_relevant_snippet(
    text,
    question,
    window=500
):
    text = str(text).replace("\n", " ")
    text_lower = text.lower()

    q = question.lower()

    # --------------------------------------------------
    # Important concepts from the question
    # --------------------------------------------------

    keywords = []

    if "total amount" in q or "total" in q:
        keywords += [
            "invoice total",
            "total amount",
            "net amount due",
            "net total",
            "grand total",
            "total:"
        ]

    if "amount due" in q:
        keywords += [
            "amount due",
            "net amount due",
            "balance due",
            "invoice total"
        ]

    if "vendor" in q:
        keywords += ["vendor", "supplier", "from"]

    if "customer" in q:
        keywords += ["customer", "bill to", "billing address"]

    if "date" in q:
        keywords += [
            "invoice date",
            "issue date",
            "date:"
        ]

    # Fallback: question words
    if not keywords:
        stop_words = {
            "what", "is", "the", "a", "an", "of",
            "in", "this", "that", "which", "for",
            "and", "to", "does", "do", "are"
        }

        keywords = [
            word
            for word in re.findall(
                r"[a-zA-Z]+",
                q
            )
            if word not in stop_words and len(word) > 2
        ]

    # --------------------------------------------------
    # Find candidate positions
    # --------------------------------------------------

    candidates = []

    for keyword in keywords:

        start = 0

        while True:

            position = text_lower.find(
                keyword.lower(),
                start
            )

            if position == -1:
                break

            candidates.append(position)

            start = position + 1

    # --------------------------------------------------
    # Also find monetary values
    # --------------------------------------------------

    money_matches = list(
        re.finditer(
            r"[$€£]\s?\d[\d,]*(?:\.\d{2})?",
            text
        )
    )

    # Prefer a keyword occurring near a money value
    best_position = None
    best_score = -1

    for position in candidates:

        nearby_money = any(
            abs(match.start() - position) <= 1200
            for match in money_matches
        )

        score = 0

        if nearby_money:
            score += 10

        # Prefer more specific phrases
        for keyword in keywords:
            if text_lower.startswith(
                keyword.lower(),
                position
            ):
                score += len(keyword)

        if score > best_score:
            best_score = score
            best_position = position

    # If no keyword found, use first money value
    if best_position is None and money_matches:
        best_position = money_matches[0].start()

    if best_position is None:
        return text[:window * 2]

    start = max(
        0,
        best_position - window
    )

    end = min(
        len(text),
        best_position + window
    )

    return text[start:end]

In [102]:
question = "What is the total amount mentioned in this invoice?"

results, context = build_compact_rag_context(
    question,
    top_k=3
)

print("Context size:", len(context), "characters")
print("\n" + context)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Context size: 2651 characters

DOCUMENT: invoice_0602
CONTENT: gency Code Estimate Number Alt Order # Friends of Chris Smith CAS October 2018 Billing Address: Order # Friends of Chris Smith Attention: Accounts Payable PO Box 3184 Hamilton, NJ 08619 732-866-9700 Billing Type Deal# Cash, Billing Gro Send Payment To: Townsquare Media - Trenton PO Box 28052 New York, NY 10087-8052 Advertiser Code Product 1/2 Summary: WKXW-FI 723631A-1 Totals: Total Product 31 October 2018 31 Property Invoice Number Spots Description Gross Total Commission Net Total Tax 1 Tax 2 Invoice Total $12,780.00 $12,780.00 $12,780.00 $12,780.00 $0.00 $12,780.00 $0.00 $0.00 $0.00 $12,780.00 $0.00 $0.00 Net Due upon Receipt Net Total $12,780.00 $0.00 Invoice Balance as of 11/05/18 1:50:58 PMI ET Invoice Detail: WKXW-FI 723631A-1 Total Product 31 October 2018 Property Invoice Number Spots Description Gross Total Commission Net Total Tax 1 Tax 2 Invoice Total $12,780.00 Rate Type $12,780.00 $0.00 $12,780.00 $0.00 $0.00 S

In [103]:
question = "What is the total amount mentioned in this invoice?"

results, context = build_compact_rag_context(
    question,
    top_k=1
)

prompt = f"""
Answer the question using ONLY the evidence.

Question:
{question}

Evidence:
{context}

Write exactly ONE short sentence.
Use proper capitalization and punctuation.
Do not invent information.
Output only the answer.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

print(answer)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

The total amount mentioned in this invoice is $12,780.00.


In [104]:
question = "What is the total amount mentioned in this invoice?"

results, context = build_compact_rag_context(
    question,
    top_k=1
)

prompt = f"""
Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{context}

Write exactly ONE short sentence.
Use proper capitalization and punctuation.
Do not invent information.
Output only the answer.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

source = results.iloc[0]["document_id"]

print(answer)
print(f"Source: {source}")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

The total amount mentioned in this invoice is $12,780.00.
Source: invoice_0602


In [105]:
question = "How is the termination fee calculated in contract_0266?"

results, context = build_compact_rag_context(
    question,
    top_k=1
)

prompt = f"""
Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{context}

Write exactly ONE short sentence.
Use proper capitalization and punctuation.
Do not invent information.
Output only the answer.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

source = results.iloc[0]["document_id"]

print(answer)
print(f"Source: {source}")

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

The Termination for Convenience Fee is calculated based on the estimated remaining value of the Developed Software.
Source: contract_0400


In [106]:
import re

def detect_document_id(question):
    match = re.search(
        r"\b(?:contract|invoice|report|email|purchase_order)_\d{4}\b",
        question.lower()
    )

    if match:
        return match.group(0)

    return None


questions = [
    "How is the termination fee calculated in contract_0266?",
    "What is the total amount in invoice_0292?",
    "What does contract_0400 say about termination?"
]

for question in questions:
    print(
        question,
        "→",
        detect_document_id(question)
    )

How is the termination fee calculated in contract_0266? → contract_0266
What is the total amount in invoice_0292? → invoice_0292
What does contract_0400 say about termination? → contract_0400


In [107]:
def retrieve_document_context(
    question,
    document_id,
    top_k=3
):
    """
    Retrieve the most relevant chunks from one specific document.
    """

    doc_chunks = full_chunks_df[
        full_chunks_df["document_id"] == document_id
    ].copy()

    if doc_chunks.empty:
        return doc_chunks, ""

    query_embedding = embedding_model.encode(
        question,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # Get embedding rows corresponding to this document
    indices = doc_chunks.index.to_numpy()

    doc_embeddings = full_embeddings[indices]

    scores = doc_embeddings @ query_embedding

    doc_chunks = doc_chunks.copy()
    doc_chunks["similarity"] = scores

    doc_chunks = (
        doc_chunks
        .sort_values("similarity", ascending=False)
        .head(top_k)
    )

    context_parts = []

    for _, row in doc_chunks.iterrows():

        context_parts.append(
            f"DOCUMENT: {row['document_id']}\n"
            f"CHUNK: {row['chunk_id']}\n"
            f"CONTENT:\n{row['text']}"
        )

    context = "\n\n".join(context_parts)

    return doc_chunks, context

In [108]:
question = "How is the termination fee calculated in contract_0266?"

document_id = detect_document_id(question)

results, context = retrieve_document_context(
    question,
    document_id,
    top_k=2
)

print("Document:", document_id)
print("Retrieved chunks:", len(results))
print("\n" + context[:5000])

Document: contract_0266
Retrieved chunks: 2

DOCUMENT: contract_0266
CHUNK: contract_0266_1
CONTENT:
(w) "Term" shall have the meaning set forth in Section 2.01. (x) "Termination Fee" shall mean an amount equal to the average Service Fees per day over the 180 day period immediately preceding the date written notice of termination is provided pursuant to Section 8.01(d) and (e) multiplied by number of days by which the Notice Period will be less than 180. (y) "Territory" means the United States, excluding its territories and possessions. 2 CONFIDENTIAL TREATMENT REQUESTED 1.02 Interpretation. The words "hereof," "herein," "hereto" and "hereunder" and words of similar import, when used in this Agreement, shall refer to this Agreement as a whole and not to any particular provision of this Agreement. The terms defined in the singular shall have a comparable meaning when used in the plural and vice versa. The term "including" shall mean "including, without limitation." When a reference is m

In [109]:
question = "How is the termination fee calculated in contract_0266?"

prompt = f"""
Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{context}

Write exactly ONE short sentence.
Use proper capitalization and punctuation.
Do not invent information.
Preserve all numeric values exactly.
Do not mention dates.
Output only the answer.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=45,
        do_sample=False,
        use_cache=True
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

source = document_id

print(answer)
print(f"Source: {source}")

The termination fee in contract_0266 is calculated based on the average Service Fees per day over the 180-day period immediately preceding the date written notice of termination is provided. 

This can be determined from
Source: contract_0266


In [110]:
def extract_exact_answer(question, context):

    q = question.lower()

    # --------------------------------------------------
    # Formula / calculation questions
    # --------------------------------------------------

    if any(word in q for word in [
        "calculate",
        "calculated",
        "calculation",
        "formula"
    ]):

        sentences = re.split(
            r'(?<=[.!?])\s+',
            context
        )

        candidates = []

        for sentence in sentences:

            s = sentence.strip()

            score = 0

            if "termination fee" in s.lower():
                score += 3

            if "multiplied" in s.lower():
                score += 4

            if "average" in s.lower():
                score += 2

            if "180" in s:
                score += 2

            if "notice period" in s.lower():
                score += 2

            if score > 0:
                candidates.append(
                    (score, s)
                )

        if candidates:

            candidates.sort(
                key=lambda x: x[0],
                reverse=True
            )

            return candidates[0][1]

    # --------------------------------------------------
    # Fallback
    # --------------------------------------------------

    return None

In [111]:
exact_answer = extract_exact_answer(
    "How is the termination fee calculated in contract_0266?",
    context
)

print(exact_answer)

(x) "Termination Fee" shall mean an amount equal to the average Service Fees per day over the 180 day period immediately preceding the date written notice of termination is provided pursuant to Section 8.01(d) and (e) multiplied by number of days by which the Notice Period will be less than 180.


In [112]:
def smart_qa(question):
    document_id = detect_document_id(question)

    if document_id:
        results, context = retrieve_document_context(
            question,
            document_id,
            top_k=2
        )
    else:
        results, context = build_compact_rag_context(
            question,
            top_k=2
        )

        if results.empty:
            return "No relevant evidence found."

        document_id = results.iloc[0]["document_id"]

    # -----------------------------------------------
    # Exact / formula questions
    # -----------------------------------------------

    exact_answer = extract_exact_answer(
        question,
        context
    )

    if exact_answer:
        print("Answer type: Extractive")
        print("Source:", document_id)
        print("\nAnswer:")
        print(exact_answer)
        return exact_answer

    # -----------------------------------------------
    # General questions → Qwen
    # -----------------------------------------------

    prompt = f"""
Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{context}

Write 1-2 short sentences.
Use proper capitalization and punctuation.
Do not invent information.
Output only the answer.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,
            use_cache=True
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    print("Answer type: LLM")
    print("Source:", document_id)
    print("\nAnswer:")
    print(answer)

    return answer

In [113]:
smart_qa(
    "How is the termination fee calculated in contract_0266?"
)

Answer type: Extractive
Source: contract_0266

Answer:
(x) "Termination Fee" shall mean an amount equal to the average Service Fees per day over the 180 day period immediately preceding the date written notice of termination is provided pursuant to Section 8.01(d) and (e) multiplied by number of days by which the Notice Period will be less than 180.


'(x) "Termination Fee" shall mean an amount equal to the average Service Fees per day over the 180 day period immediately preceding the date written notice of termination is provided pursuant to Section 8.01(d) and (e) multiplied by number of days by which the Notice Period will be less than 180.'

In [114]:
smart_qa(
    "What happens if the Company gives less than six months notice in contract_0266?"
)

Answer type: LLM
Source: contract_0266

Answer:
If the company provides less than six months notice, they must pay the contractor an amount equal to the termination fee. Answer: If the company provides less than six months notice, they must pay the contractor


'If the company provides less than six months notice, they must pay the contractor an amount equal to the termination fee. Answer: If the company provides less than six months notice, they must pay the contractor'

In [115]:
prompt = f"""
Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{context}

Write exactly ONE short sentence.
Use proper capitalization and punctuation.
Do not invent information.
Do not write "Answer:".
Do not repeat the answer.
Output only the sentence.
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=35,
        do_sample=False,
        use_cache=True,
        no_repeat_ngram_size=3
    )

answer = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
).strip()

# Remove accidental continuation
if "Answer:" in answer:
    answer = answer.split("Answer:")[0].strip()

# Remove surrounding quotes
answer = answer.strip('"').strip("'").strip()

print("Answer type: LLM")
print("Source:", document_id)
print("\nAnswer:")
print(answer)

Answer type: LLM
Source: contract_0266

Answer:
The termination fee in contract 0276 is calculated based on the average service fees per day in the 90-day period immediately before the notice oftermination is given


In [116]:
def extract_conditional_answer(question, context):
    """
    Extract the sentence(s) that describe what happens
    under a condition in the question.
    """

    q = question.lower()
    context = str(context).replace("\n", " ")

    sentences = re.split(
        r'(?<=[.!?])\s+',
        context
    )

    # Important terms from the question
    question_words = [
        word
        for word in re.findall(r"[a-zA-Z]+", q)
        if len(word) > 2
    ]

    candidates = []

    for sentence in sentences:

        s = sentence.strip()
        s_lower = s.lower()

        score = 0

        # Match important question concepts
        for word in question_words:

            if word in s_lower:
                score += 1

        # Strong signals for conditional/payment questions
        if "if" in s_lower:
            score += 2

        if "provided" in s_lower:
            score += 2

        if "less than" in s_lower:
            score += 3

        if "must pay" in s_lower:
            score += 3

        if "shall pay" in s_lower:
            score += 3

        if "pay contractor" in s_lower:
            score += 3

        if "termination fee" in s_lower:
            score += 2

        if score > 0:
            candidates.append(
                (score, s)
            )

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return candidates[0][1]

In [117]:
question = "What happens if the Company gives less than six months notice in contract_0266?"

document_id = detect_document_id(question)

results, context = retrieve_document_context(
    question,
    document_id,
    top_k=2
)

answer = extract_conditional_answer(
    question,
    context
)

print("Answer type: Extractive")
print("Source:", document_id)
print("\nAnswer:")
print(answer)

Answer type: Extractive
Source: contract_0266

Answer:
The Company may terminate this Agreement in accordance with the immediately preceding sentence but with less than six (6) months' prior written notice to Contractor; provided, that in such event, the Company shall pay Contractor an amount equal to the Termination Fee.


In [118]:
def smart_qa(question):
    """
    General DocuMind QA router.

    Exact/formula questions  -> Extractive
    Conditional questions   -> Extractive
    Other questions         -> Qwen
    """

    document_id = detect_document_id(question)

    # --------------------------------------------------
    # 1. Retrieve context
    # --------------------------------------------------

    if document_id:

        results, context = retrieve_document_context(
            question,
            document_id,
            top_k=2
        )

    else:

        results, context = build_compact_rag_context(
            question,
            top_k=2
        )

        if results.empty:
            return "No relevant evidence found."

        document_id = results.iloc[0]["document_id"]

    if results.empty:
        return "No relevant evidence found."

    # --------------------------------------------------
    # 2. Exact / formula questions
    # --------------------------------------------------

    exact_answer = extract_exact_answer(
        question,
        context
    )

    if exact_answer:

        print("Answer type: Extractive")
        print("Source:", document_id)
        print("\nAnswer:")
        print(exact_answer)

        return exact_answer

    # --------------------------------------------------
    # 3. Conditional questions
    # --------------------------------------------------

    q = question.lower()

    conditional = (
        "what happens if" in q
        or "what happens when" in q
        or "what if" in q
        or "what is required if" in q
        or "if the" in q
    )

    if conditional:

        conditional_answer = extract_conditional_answer(
            question,
            context
        )

        if conditional_answer:

            print("Answer type: Extractive")
            print("Source:", document_id)
            print("\nAnswer:")
            print(conditional_answer)

            return conditional_answer

    # --------------------------------------------------
    # 4. General questions → Qwen
    # --------------------------------------------------

    prompt = f"""
Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{context}

Write exactly ONE short sentence.
Use proper capitalization and punctuation.
Do not invent information.
Do not change names, numbers, or values.
Do not write "Answer:".
Output only the answer.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=35,
            do_sample=False,
            use_cache=True
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    if "Answer:" in answer:
        answer = answer.split("Answer:")[0].strip()

    answer = answer.strip('"').strip("'").strip()

    print("Answer type: LLM")
    print("Source:", document_id)
    print("\nAnswer:")
    print(answer)

    return answer

In [119]:
smart_qa(
    "How is the termination fee calculated in contract_0266?"
)

print("\n" + "=" * 80 + "\n")

smart_qa(
    "What happens if the Company gives less than six months notice in contract_0266?"
)

Answer type: Extractive
Source: contract_0266

Answer:
(x) "Termination Fee" shall mean an amount equal to the average Service Fees per day over the 180 day period immediately preceding the date written notice of termination is provided pursuant to Section 8.01(d) and (e) multiplied by number of days by which the Notice Period will be less than 180.


Answer type: Extractive
Source: contract_0266

Answer:
The Company may terminate this Agreement in accordance with the immediately preceding sentence but with less than six (6) months' prior written notice to Contractor; provided, that in such event, the Company shall pay Contractor an amount equal to the Termination Fee.


"The Company may terminate this Agreement in accordance with the immediately preceding sentence but with less than six (6) months' prior written notice to Contractor; provided, that in such event, the Company shall pay Contractor an amount equal to the Termination Fee."

In [120]:
_ = smart_qa(
    "What happens if the Company gives less than six months notice in contract_0266?"
)

Answer type: Extractive
Source: contract_0266

Answer:
The Company may terminate this Agreement in accordance with the immediately preceding sentence but with less than six (6) months' prior written notice to Contractor; provided, that in such event, the Company shall pay Contractor an amount equal to the Termination Fee.


In [121]:
rag_eval_questions = [
    "What is the total amount mentioned in this invoice?",
    "How is the termination fee calculated in contract_0266?",
    "What happens if the Company gives less than six months notice in contract_0266?",
    "Which contracts mention early termination fees?",
    "What does this purchase order contain?"
]

for question in rag_eval_questions:

    print("\n" + "=" * 100)
    print("QUESTION:", question)

    try:
        _ = smart_qa(question)
    except Exception as e:
        print("ERROR:", type(e).__name__, "-", e)


QUESTION: What is the total amount mentioned in this invoice?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Answer type: LLM
Source: invoice_0602

Answer:
The total amount mentioned in this invoice is $12,780.00.

QUESTION: How is the termination fee calculated in contract_0266?
Answer type: Extractive
Source: contract_0266

Answer:
(x) "Termination Fee" shall mean an amount equal to the average Service Fees per day over the 180 day period immediately preceding the date written notice of termination is provided pursuant to Section 8.01(d) and (e) multiplied by number of days by which the Notice Period will be less than 180.

QUESTION: What happens if the Company gives less than six months notice in contract_0266?
Answer type: Extractive
Source: contract_0266

Answer:
The Company may terminate this Agreement in accordance with the immediately preceding sentence but with less than six (6) months' prior written notice to Contractor; provided, that in such event, the Company shall pay Contractor an amount equal to the Termination Fee.

QUESTION: Which contracts mention early termination fees?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Answer type: LLM
Source: contract_0400

Answer:
The contract mentions an early termination fee when Customer terminates the Agreement for convenience, as specified in DOCUMENT: contract_0400.

QUESTION: What does this purchase order contain?


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Answer type: LLM
Source: po_0022

Answer:
The purchase order contains details for a radio station's spot advertising campaign from HQU H.Q. Energy Services on WBLM-F in Portland, Maine, including estimated costs and


In [124]:
def smart_qa(question):
    """
    General DocuMind QA router.

    Routes:
    1. Termination enumeration -> deterministic grounded pipeline
    2. Exact/formula questions -> extractive
    3. Conditional questions -> extractive
    4. Other questions -> Qwen
    """

    q = question.lower()

    # --------------------------------------------------
    # 1. Termination enumeration
    # --------------------------------------------------

    termination_words = [
        "termination fee",
        "early termination fee",
        "termination charge",
        "termination penalty",
        "liquidated damages"
    ]

    is_termination_query = any(
        word in q
        for word in termination_words
    )

    is_enumeration = (
        "which contracts" in q
        or "what contracts" in q
        or "contracts mention" in q
        or "contracts contain" in q
    )

    if is_termination_query and is_enumeration:

        answer = answer_question(question)

        print("\nAnswer type: Grounded")
        print(answer)

        return answer

    # --------------------------------------------------
    # 2. Detect specific document
    # --------------------------------------------------

    document_id = detect_document_id(question)

    if document_id:

        results, context = retrieve_document_context(
            question,
            document_id,
            top_k=2
        )

    else:

        results, context = build_compact_rag_context(
            question,
            top_k=2
        )

        if results.empty:
            return "No relevant evidence found."

        document_id = results.iloc[0]["document_id"]

    if results.empty:
        return "No relevant evidence found."

    # --------------------------------------------------
    # 3. Exact / formula questions
    # --------------------------------------------------

    exact_answer = extract_exact_answer(
        question,
        context
    )

    if exact_answer:

        print("Answer type: Extractive")
        print("Source:", document_id)
        print("\nAnswer:")
        print(exact_answer)

        return exact_answer

    # --------------------------------------------------
    # 4. Conditional questions
    # --------------------------------------------------

    conditional = (
        "what happens if" in q
        or "what happens when" in q
        or "what if" in q
        or "what is required if" in q
    )

    if conditional:

        conditional_answer = extract_conditional_answer(
            question,
            context
        )

        if conditional_answer:

            print("Answer type: Extractive")
            print("Source:", document_id)
            print("\nAnswer:")
            print(conditional_answer)

            return conditional_answer

    # --------------------------------------------------
    # 5. General questions -> Qwen
    # --------------------------------------------------

    prompt = f"""
Answer the question using ONLY the evidence below.

Question:
{question}

Evidence:
{context}

Write exactly ONE short sentence.
Maximum 20 words.
Use proper capitalization and punctuation.
Do not invent information.
Do not change names, numbers, or values.
Do not write "Answer:".
Output only the answer.
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=45,
            do_sample=False,
            use_cache=True
        )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    if "Answer:" in answer:
        answer = answer.split("Answer:")[0].strip()

    answer = answer.strip('"').strip("'").strip()

    print("Answer type: LLM")
    print("Source:", document_id)
    print("\nAnswer:")
    print(answer)

    return answer

In [123]:
_ = smart_qa(
    "Which contracts mention early termination fees?"
)

print("\n" + "=" * 100 + "\n")

_ = smart_qa(
    "How is the termination fee calculated in contract_0266?"
)

print("\n" + "=" * 100 + "\n")

_ = smart_qa(
    "What does this purchase order contain?"
)

QUESTION: Which contracts mention early termination fees?
Exhaustive scan: 6069 chunks
Requested types: ['Early termination fee']
Verified contracts: 4

Answer type: Grounded
The evidence identifies 4 contracts matching the requested termination-related terminology. The contracts are: contract_0086, contract_0135, contract_0302, contract_0400.


Answer type: Extractive
Source: contract_0266

Answer:
(x) "Termination Fee" shall mean an amount equal to the average Service Fees per day over the 180 day period immediately preceding the date written notice of termination is provided pursuant to Section 8.01(d) and (e) multiplied by number of days by which the Notice Period will be less than 180.




Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Answer type: LLM
Source: po_0022

Answer:
The purchase order contains details for a radio station's spot advertising campaign from HQU Energy Services on WBLM-F in Portland, Maine, including estimated


In [125]:
_ = smart_qa(
    "What does this purchase order contain?"
)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

Answer type: LLM
Source: po_0022

Answer:
The purchase order contains details for a radio station's spot advertising campaign from HQU Energy Services on WBLM-F in Portland, Maine, including estimated costs and delivery dates. The document also includes confirmation of an existing purchase order
